# Preprocesamiento

## 1. Carga del archivo

In [1]:
import pyreadstat
import re
import unicodedata
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import prince
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

df, meta = pyreadstat.read_sav("/Users/tomas/github/eaui_subtel/data/sav/2026.sav")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

Filas: 5,000 | Columnas: 587


## 2. GSE derivado

In [2]:
def _educ_g(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 3:  return 'basica'
    if e <= 7:  return 'media'
    if e <= 9:  return 'tecnica'
    return 'universitaria'

_M = {
    (1,'basica'):'E',  (1,'media'):'E',  (1,'tecnica'):'D',  (1,'universitaria'):'D',
    (2,'basica'):'E',  (2,'media'):'D',  (2,'tecnica'):'D',  (2,'universitaria'):'C3',
    (3,'basica'):'D',  (3,'media'):'C3', (3,'tecnica'):'C3', (3,'universitaria'):'C2',
    (4,'basica'):'C3', (4,'media'):'C2', (4,'tecnica'):'C2', (4,'universitaria'):'C1',
    (5,'basica'):'C2', (5,'media'):'C1', (5,'tecnica'):'C1', (5,'universitaria'):'AB',
    (6,'basica'):'C1', (6,'media'):'AB', (6,'tecnica'):'AB', (6,'universitaria'):'AB',
}
_ORDEN_GSE = ['AB','C1','C2','C3','D','E']  # Invertido

_eg = df['A10'].apply(_educ_g)
df['gse'] = pd.Categorical(
    df['A11'].combine(_eg, lambda o, e: np.nan if pd.isna(o) or e is None else _M.get((int(o), e), np.nan)),
    categories=_ORDEN_GSE, ordered=True  # Aquí usa el orden invertido automáticamente
)
print('GSE:', df['gse'].value_counts().reindex(_ORDEN_GSE).to_dict())

GSE: {'AB': 342, 'C1': 533, 'C2': 988, 'C3': 1316, 'D': 833, 'E': 988}


## 3. Etiquetas limpias

In [3]:
def limpiar_etiqueta(label):
    """Extrae la parte descriptiva útil de una etiqueta SPSS."""
    if not label: return label
    label = label.strip()
    # Patrón B/C: empieza con código de variable (P3_1 .-, Q1.3.-)
    if re.match(r'^[A-Z]\w+[\._]\w+\s*\.-?', label):
        if ':' in label:
            r = label.split(':')[-1].strip()
            if r: return r
        if '?' in label:
            r = label.split('?')[-1].strip().lstrip(':').strip()
            if r: return r
        r = re.sub(r'^[A-Z]\w+[\._]\w+[\s\._\-]+', '', label).strip()
        return r.lstrip('.-').strip()
    # Patrón A: etiqueta + [pregunta padre] (corchete abre, cierre o no)
    if '[' in label:
        r = label[:label.index('[')].strip()
        return re.sub(r'^\d+[\.-]+\s*', '', r).strip()
    # Patrón D: numeración inicial
    return re.sub(r'^\d+[\.-]+\s*', '', label).strip()


etiquetas_limpias = {
    col: limpiar_etiqueta(label)
    for col, label in zip(meta.column_names, meta.column_labels) if label
}
print(f"Etiquetas procesadas: {len(etiquetas_limpias)}")

Etiquetas procesadas: 587


## 4. Diccionario de variables originales

In [4]:
diccionario = pd.DataFrame({'variable': meta.column_names, 'etiqueta': meta.column_labels})
diccionario


,variable,etiqueta
0,REGISTRO,Número de registro
1,FECHAFIN,Fecha de fin de la entrevista
2,COD_REGION,Región
3,COMUNA_DEF,Comuna
4,ZONA,ZONA
...,...,...
582,A12_1,A12.- ¿a cuál de los siguientes tramos corresp...
583,POND_HOGAR,Ponderador Hogar
584,FE_HOGAR,Factor de Expansión Hogar
585,POND_PERSONAS,PONDORADOR PERSONAS


In [5]:
# Buscar variable por nombre o fragmento de etiqueta
busqueda = 'Q1'
diccionario[
    diccionario['variable'].str.contains(busqueda, case=False) |
    diccionario['etiqueta'].str.contains(busqueda, case=False, na=False)
]

,variable,etiqueta
177,Q1,Q1.- ¿Cuál es su parentesco con el Jefe /a de ...
178,Q1_1,Q1_1.- ¿Cuál es su edad?
179,Q1_2,Q1_2.- Sexo del entrevistado
180,Q1_3,Q1.3.- ¿Cuál fue el último curso y tipo de est...
181,Q1_3_1,Q1.3.- ¿Cuál fue el último curso y tipo de est...
182,Q1_4,Q1_4.- ¿Cuál es su profesión o trabajo o activ...
183,Q1_4_1,Q1.3.- ¿Cuál fue el último curso y tipo de est...
184,Q1_5_1,Aymara [Q1.5.- ¿Usted pertenece o es descendie...
185,Q1_5_2,Rapa-Nui [Q1.5.- ¿Usted pertenece o es descend...
186,Q1_5_3,Quechua [Q1.5.- ¿Usted pertenece o es descendi...


In [6]:
# Ver categorías codificadas de una variable
variable = 'A9'
labels = meta.variable_value_labels.get(variable, {})
if labels:
    for k, v in labels.items(): print(f'  {k} -> {v}')
else:
    print(f"'{variable}' no tiene etiquetas de valor.")

  1.0 -> Yo soy el jefe de Hogar
  2.0 -> Cónyuge o Pareja
  3.0 -> Hijo(a)
  4.0 -> Yerno o Nuera
  5.0 -> Nieto(a)
  6.0 -> Padre o Madre
  7.0 -> Suegro(a)
  8.0 -> Hermano(a)
  9.0 -> Cuñado(a)
  10.0 -> Otro familiar
  11.0 -> Otro no familiar
  12.0 -> Servicio Doméstico


## 5. Tratamiento NS/NR

Variables de montos usan `9999999` como código NS/NR.

In [7]:
cols_nsnr = [
    'P11','Q7_4',
    'P17_1','P17_2','P17_3','P17_4','P17_5',
    'P19_1','P19_2','P19_3','P19_4',
    'Q40_1','Q40_2','Q40_3','Q40_4','Q40_5',
    'Q42','Q42_1'
]
for col in cols_nsnr:
    if col in df.columns: df[col] = df[col].replace(9999999, np.nan)
print('NS/NR reemplazados por NaN.')

NS/NR reemplazados por NaN.


## 6. Renombrado de variables

In [8]:
nombres_cortos = {
    'REGISTRO':'id', 'FECHAFIN':'fecha_fin', 'COD_REGION':'region', 'COMUNA_DEF':'comuna', 'ZONA':'zona',
    'A9':'parentesco_jh', 'A10':'educ_jh', 'A11':'ocupacion_jh', 'A12_1':'ingreso_hogar',
    'Q1':'parentesco', 'Q1_1':'edad', 'Q1_2':'sexo', 'Q1_3':'educ', 'Q1_4':'ocupacion_encuestado', 'Q2':'actividad',
    'P1':'acceso_internet_hogar', 'P2':'n_smartphones_hogar', 'P2_1':'n_computadores_hogar',
    'P10':'tipo_acceso_fijo', 'P11':'pago_mensual_internet', 'P11_3':'velocidad_contratada',
    'P11_4':'calidad_acceso', 'P11_5':'cuota_mensual_gb', 'P12_2':'tipo_plan', 'P12_1':'plan_movil_tipo',
    'P14':'razon_no_acceso_principal', 'P15':'disposicion_contratar_fijo',
    'Q5':'uso_computador', 'Q7':'uso_smartphone', 'Q7_1':'smartphone_propio',
    'Q7_3':'plan_movil_tipo_ind', 'Q7_4':'pago_mensual_movil',
    'Q9':'ultimo_uso_internet', 'Q10':'frecuencia_internet', 'Q11':'tiempo_diario_internet',
    'Q13':'tipo_acceso_mas_usado', 'Q14':'uso_internet_hogar', 'Q15':'frecuencia_internet_hogar',
    'Q16':'tiempo_diario_hogar', 'Q17':'uso_internet_fuera_hogar', 'Q18':'frecuencia_fuera_hogar',
    'Q19':'tiempo_diario_fuera_hogar',
    'Q23':'internet_facilita_trabajo', 'Q25':'internet_mejora_vida', 'Q27':'ultima_compra_online',
    'Q31':'percepcion_proteccion', 'Q30_1':'reg_control_legal', 'Q30_2':'reg_control_familia', 'Q30_3':'reg_autocontrol',
    'FE_HOGAR':'fe_hogar', 'FE_PERSONAS':'fe_personas', 'POND_HOGAR':'pond_hogar', 'POND_PERSONAS':'pond_personas',
}

df = df.rename(columns={k: v for k, v in nombres_cortos.items() if k in df.columns})

# Recodificación de educ_jh y ocupacion_jh (aquí, con valores numéricos aún intactos)
_mapa_educ = {
    1:'Sin educación formal', 2:'Básica incompleta', 3:'Básica completa',
    4:'Media CH incompleta', 5:'Media TP incompleta', 6:'Media CH completa', 7:'Media TP completa',
    8:'Superior técnica incompleta', 9:'Superior técnica completa',
    10:'Superior universitaria incompleta', 11:'Superior universitaria completa'
}
_mapa_ocup = {
    1:'Trabajos ocasionales e informales', 2:'Oficio menor - obrero no calificado',
    3:'Obrero calificado - microempresario', 4:'Empleado medio - técnico - prof. independiente',
    5:'Ejecutivo medio - prof. universitario', 6:'Alto ejecutivo - empresario - directivo'
}
df['educ_jh']              = df['educ_jh'].map(_mapa_educ)
df['ocupacion_jh']         = df['ocupacion_jh'].map(_mapa_ocup)
df['ocupacion_encuestado'] = df['ocupacion_encuestado'].map({**_mapa_ocup, 7:'Sin trabajo remunerado'})

print(f"Renombradas: {len(nombres_cortos)} | Columnas totales: {df.shape[1]}")

Renombradas: 53 | Columnas totales: 588


## 7. Recodificaciones

In [9]:
# ── Recoding: Habilidades Digitales Q8 (0='No', 1='Sí') ──────────────

q8_vars = [
    'Q8_10', 'Q8_11', 'Q8_12', 'Q8_13', 'Q8_15', 'Q8_16', 'Q8_1', 'Q8_2',
    'Q8_3', 'Q8_4', 'Q8_5', 'Q8_6', 'Q8_14', 'Q8_17', 'Q8_18', 'Q8_7',
    'Q8_8', 'Q8_9'
]

for col in q8_vars:
    if col in df.columns:
        df[col] = df[col].replace({0: 'No', 1: 'Sí'})

print(f'✓ Recoding Q8: 0→No, 1→Sí | {len(q8_vars)} variables procesadas')
print(f'  Ejemplo {q8_vars[0]}: {df[q8_vars[0]].value_counts().to_dict()}')

✓ Recoding Q8: 0→No, 1→Sí | 18 variables procesadas
  Ejemplo Q8_10: {'No': 2754, 'Sí': 2004}


In [10]:
df = df.copy()

# Identificación
df['region'] = df['region'].map({
    1:'Tarapacá', 2:'Antofagasta', 3:'Atacama', 4:'Coquimbo', 5:'Valparaíso',
    6:"O'Higgins", 7:'Maule', 8:'Biobío', 9:'Araucanía', 10:'Los Lagos',
    11:'Aysén', 12:'Magallanes', 13:'Metropolitana', 14:'Los Ríos', 15:'Arica y Parinacota', 16:'Ñuble'
})
df['zona'] = df['zona'].map({1:'Urbana', 2:'Rural'})

# Sociodemográficas del entrevistado
df['sexo'] = df['sexo'].map({1:'Hombre', 2:'Mujer'})
df['educ'] = df['educ'].map(_mapa_educ)
df['educ_grupo'] = df['educ'].map({
    'Sin educación formal':'Básica o menos', 'Básica incompleta':'Básica o menos',
    'Básica completa':'Básica o menos', 'Media CH incompleta':'Media',
    'Media TP incompleta':'Media', 'Media CH completa':'Media', 'Media TP completa':'Media',
    'Superior técnica incompleta':'Superior', 'Superior técnica completa':'Superior',
    'Superior universitaria incompleta':'Superior', 'Superior universitaria completa':'Superior',
})
df['tramo_edad'] = pd.cut(df['edad'], bins=[0,17,29,44,59,200],
                          labels=['Menor de 18','18-29','30-44','45-59','60 y más'], right=True)
df['actividad'] = df['actividad'].map({
    1:'Trabajador independiente', 2:'Empleador/patrón', 3:'Empleado dependiente',
    4:'Familiar no remunerado', 5:'FFAA y de orden', 6:'Cesante',
    7:'Jubilado/pensionado', 8:'Estudiante', 9:'Labores del hogar'
})

# Acceso a internet
df['acceso_internet_hogar'] = df['acceso_internet_hogar'].map({1:'Sí', 2:'No'})
df['tipo_acceso_fijo'] = df['tipo_acceso_fijo'].map({
    1:'ADSL', 2:'Cable/Módem', 3:'Fibra óptica', 4:'Inalámbrica',
    5:'Satelital', 31:'WiFi', 32:'Antena', 33:'Banda ancha', 34:'Acceso telefónico', 88:'No sabe'
})
df['velocidad_contratada'] = df['velocidad_contratada'].map({
    1:'Hasta 10 Mbps', 2:'Más de 10 a 100 Mbps', 3:'Más de 100 a 500 Mbps',
    4:'Más de 500 Mbps a 1 Gbps', 5:'Más de 1 Gbps', 99:'NS/NR'
})
df['tipo_plan'] = df['tipo_plan'].map({
    1:'Banda ancha desnuda', 2:'BA + TV Cable', 3:'BA + Telefonía fija',
    4:'Triple pack (BA+TV+Tel)', 5:'Otros planes'
})
df['tipo_acceso_mas_usado'] = df['tipo_acceso_mas_usado'].map({
    1.0:'Banda Ancha Fija / WiFi', 2.0:'Banda Ancha Móvil',
    3.0:'Internet Móvil (Smartphone/Tablet)', 4.0:'Conexión Satelital'
})

# Uso individual
df['uso_computador']  = df['uso_computador'].map({1:'Sí', 2:'No'})
df['uso_smartphone']  = df['uso_smartphone'].map({1:'Sí', 2:'No'})
df['ultimo_uso_internet'] = df['ultimo_uso_internet'].map({
    1:'Hoy', 2:'Entre 2 y 3 días', 3:'Entre 3 y 7 días', 4:'Entre 1 y 4 semanas',
    5:'Más de 4 semanas', 6:'Más de 12 meses', 7:'Nunca'
})
df['frecuencia_internet'] = df['frecuencia_internet'].map({
    1:'Todos los días', 2:'Varias veces por semana',
    3:'Al menos una vez al mes', 4:'Menos de una vez al mes'
})
df['tiempo_diario_internet'] = df['tiempo_diario_internet'].map({
    1:'Menos de 1 hora', 2:'Entre 1 y 2 horas', 3:'Entre 2 y 4 horas', 4:'Más de 4 horas'
})

# Percepciones
df['percepcion_proteccion']     = df['percepcion_proteccion'].map({
    1:'Muy protegido', 2:'Protegido', 3:'Desprotegido', 4:'Muy desprotegido', 99:'NS/NR'
})
df['internet_mejora_vida']      = df['internet_mejora_vida'].map({1:'Sí', 2:'No'})
df['internet_facilita_trabajo'] = df['internet_facilita_trabajo'].map({1:'Sí', 2:'No'})

print('Recodificaciones completadas.')
print(f"sexo: {df['sexo'].value_counts().to_dict()}")
print(f"acceso: {df['acceso_internet_hogar'].value_counts().to_dict()}")

Recodificaciones completadas.
sexo: {'Mujer': 2815, 'Hombre': 2185}
acceso: {'Sí': 4841, 'No': 159}


## 8. Ingreso del hogar

In [11]:
_rangos = {
    11:(0,129000),12:(130000,226000),13:(227000,393000),14:(394000,686000),15:(687000,1100000),16:(1200000,2000000),17:(2100000,None),
    21:(0,210000),22:(211000,366000),23:(367000,639000),24:(640000,1100000),25:(1200000,1900000),26:(2000000,3300000),27:(3400000,None),
    31:(0,279000),32:(280000,487000),33:(488000,849000),34:(850000,1400000),35:(1500000,2500000),36:(2600000,4500000),37:(4600000,None),
    41:(0,341000),42:(342000,595000),43:(596000,1000000),44:(1100000,1800000),45:(1900000,3100000),46:(3200000,5500000),47:(5600000,None),
    51:(0,399000),52:(400000,696000),53:(697000,1200000),54:(1300000,2100000),55:(2200000,3600000),56:(3700000,6400000),57:(6500000,None),
    61:(0,453000),62:(454000,791000),63:(792000,1300000),64:(1400000,2400000),65:(2500000,4100000),66:(4200000,7300000),67:(7400000,None),
    71:(0,505000),72:(506000,881000),73:(882000,1500000),74:(1600000,2600000),75:(2700000,4600000),76:(4700000,8100000),77:(8200000,None),
    81:(0,555000),82:(556000,967000),83:(968000,1600000),84:(1700000,2900000),85:(3000000,5100000),86:(5200000,8900000),87:(9000000,None),
    91:(0,602000),92:(603000,1000000),93:(1100000,1800000),94:(1900000,3100000),95:(3200000,5500000),96:(5600000,9700000),97:(9800000,None),
    101:(0,648000),102:(649000,1100000),103:(1200000,1900000),104:(2000000,3400000),105:(3500000,5900000),106:(6000000,10400000),107:(10500000,None),
}
_mapa_pm = {float(k): (v[0]*1.5 if v[1] is None else (v[0]+v[1])/2) for k, v in _rangos.items()}

df['ingreso_pm'] = df['ingreso_hogar'].map(_mapa_pm)
df['ingreso_tramo'] = pd.cut(
    df['ingreso_pm'],
    bins=[0, 384000, 540000, 798000, 1100000, 1700000, float('inf')],
    labels=['Hasta $384 mil','$384 mil a $540 mil','$540 mil a $798 mil',
            '$798 mil a $1,1 millón','$1,1 millón a $1,7 millones','Más de $1,7 millones'],
    right=True
)
df['ingreso_grupo'] = df['ingreso_tramo'].map({
    'Hasta $384 mil':'Bajo', '$384 mil a $540 mil':'Bajo',
    '$540 mil a $798 mil':'Medio', '$798 mil a $1,1 millón':'Medio',
    '$1,1 millón a $1,7 millones':'Alto', 'Más de $1,7 millones':'Alto',
})

# Validación: promedio de ingreso debe subir de E a AB
(
    df.groupby('gse', observed=True)['ingreso_pm']
    .agg(N='count', Promedio='mean').reindex(_ORDEN_GSE).round(0).astype({'N':int,'Promedio':int})
)

,N,Promedio
gse,,
AB,286,2097505
C1,444,1389884
C2,826,986176
C3,1112,799533
D,704,650022
E,846,539833


## 9. Funciones de análisis ponderado

In [12]:
ORDEN_CATEGORIAS = {
    'sexo':         ['Hombre','Mujer'],
    'zona':         ['Urbana','Rural'],
    'region':       ['Tarapacá','Antofagasta','Atacama','Coquimbo','Valparaíso',"O'Higgins",'Maule',
                     'Biobío','Araucanía','Los Lagos','Aysén','Magallanes','Metropolitana',
                     'Los Ríos','Arica y Parinacota','Ñuble'],
    'educ':         ['Sin educación formal','Básica incompleta','Básica completa',
                     'Media CH incompleta','Media TP incompleta','Media CH completa','Media TP completa',
                     'Superior técnica incompleta','Superior técnica completa',
                     'Superior universitaria incompleta','Superior universitaria completa'],
    'educ_grupo':   ['Básica o menos','Media','Superior'],
    'tramo_edad':   ['Menor de 18','18-29','30-44','45-59','60 y más'],
    'actividad':    ['Trabajador independiente','Empleador/patrón','Empleado dependiente',
                     'Familiar no remunerado','FFAA y de orden','Cesante',
                     'Jubilado/pensionado','Estudiante','Labores del hogar'],
    'ocupacion_jh': ['Trabajos ocasionales e informales','Oficio menor - obrero no calificado',
                     'Obrero calificado - microempresario','Empleado medio - técnico - prof. independiente',
                     'Ejecutivo medio - prof. universitario','Alto ejecutivo - empresario - directivo'],
    'ocupacion_encuestado': ['Trabajos ocasionales e informales','Oficio menor - obrero no calificado',
                     'Obrero calificado - microempresario','Empleado medio - técnico - prof. independiente',
                     'Ejecutivo medio - prof. universitario','Alto ejecutivo - empresario - directivo',
                     'Sin trabajo remunerado'],
    'gse':              ['AB', 'C1', 'C2', 'C3', 'D', 'E'],
    'ingreso_tramo':    ['Hasta $384 mil','$384 mil a $540 mil','$540 mil a $798 mil',
                         '$798 mil a $1,1 millón','$1,1 millón a $1,7 millones','Más de $1,7 millones'],
    'ingreso_grupo':    ['Bajo','Medio','Alto'],
    'acceso_internet_hogar':    ['Sí','No'],
    'uso_computador':           ['Sí','No'],
    'uso_smartphone':           ['Sí','No'],
    'internet_mejora_vida':     ['Sí','No'],
    'internet_facilita_trabajo':['Sí','No'],
    'tipo_acceso_fijo':         ['Fibra óptica','Cable/Módem','ADSL','Inalámbrica','Satelital','WiFi','Antena','Banda ancha','Acceso telefónico','No sabe'],
    'tipo_acceso_mas_usado':    ['Banda Ancha Fija / WiFi','Banda Ancha Móvil','Internet Móvil (Smartphone/Tablet)','Conexión Satelital'],
    'tipo_plan':                ['Banda ancha desnuda','BA + TV Cable','BA + Telefonía fija','Triple pack (BA+TV+Tel)','Otros planes'],
    'ultimo_uso_internet':      ['Hoy','Entre 2 y 3 días','Entre 3 y 7 días',
                                 'Entre 1 y 4 semanas','Más de 4 semanas','Más de 12 meses','Nunca'],
    'frecuencia_internet':      ['Todos los días','Varias veces por semana',
                                 'Al menos una vez al mes','Menos de una vez al mes'],
    'tiempo_diario_internet':   ['Menos de 1 hora','Entre 1 y 2 horas','Entre 2 y 4 horas','Más de 4 horas'],
    'percepcion_proteccion':    ['Muy protegido','Protegido','Desprotegido','Muy desprotegido','NS/NR'],
    'velocidad_contratada':     ['Hasta 10 Mbps','Más de 10 a 100 Mbps','Más de 100 a 500 Mbps',
                                 'Más de 500 Mbps a 1 Gbps','Más de 1 Gbps','NS/NR'],
}


def fordf(df_tabla, titulo=None):
    """Formato visual: porcentajes con 1 decimal."""
    num_cols = df_tabla.select_dtypes(include=['number']).columns
    estilo = df_tabla.style.format({col: '{:.1f}' for col in num_cols})
    if titulo:
        estilo = estilo.set_caption(titulo)
    return estilo


def _ordenar(df_res, var, cruzada=False):
    """Reordena filas según ORDEN_CATEGORIAS. 'Total' siempre al final si existe."""
    if var not in ORDEN_CATEGORIAS:
        return df_res
    orden = ORDEN_CATEGORIAS[var]

    if cruzada:
        # El índice ya es la variable → reordenar filas directamente
        ok  = [v for v in orden if v in df_res.index]
        rst = [v for v in df_res.index if v not in ok and v != 'Total']
        # FIX 3: solo agregar 'Total' al reindex si efectivamente existe en el índice
        total_fila = ['Total'] if 'Total' in df_res.index else []
        return df_res.reindex(ok + rst + total_fila)

    # La variable es una columna → ordenar con pd.Categorical
    ok  = [v for v in orden if v in df_res[var].values]
    rst = [v for v in df_res[var].values if v not in ok and v != 'Total']
    # FIX 3: solo agregar 'Total' como categoría si existe en los datos
    total_cats = ['Total'] if 'Total' in df_res[var].values else []
    df_res[var] = pd.Categorical(df_res[var], categories=ok + rst + total_cats, ordered=True)
    return df_res.sort_values(var).reset_index(drop=True)


def dstats(data_df, variables, tipo='frecuencia', cruce=None, factor=None,
           transponer=False, estilo=True):
    """
    Análisis ponderado — solo porcentajes.
    Nota: NaN excluidos (porcentajes sobre casos válidos, como SPSS).
    """
    if isinstance(variables, str):
        variables = [variables]

    # FIX 1: excluir None de la lista de columnas a validar
    cols_a_verificar = variables + ([factor] if factor else []) + ([cruce] if cruce else [])
    for col in cols_a_verificar:
        if col not in data_df.columns:
            raise ValueError(f"Columna '{col}' no existe en el DataFrame.")

    # FIX 2: validar que cruce esté definido cuando tipo='cruzada'
    if tipo == 'cruzada' and cruce is None:
        raise ValueError("Para tipo='cruzada' debes especificar el parámetro 'cruce'.")

    # --- FRECUENCIA ---
    if tipo == 'frecuencia':
        var = variables[0]
        df_valid = data_df[data_df[var].notna()]
        tot = df_valid[factor].sum()
        res = (df_valid
               .groupby(var, observed=True)[factor]
               .sum()
               .reset_index()
               .rename(columns={factor: 'porcentaje'}))
        res['porcentaje'] = (res['porcentaje'] / tot * 100).round(2)
        res = _ordenar(res, var).set_index(var)

        if estilo:
            titulo = f"Frecuencia: '{var}' — base ponderada: {int(tot):,} ({factor})"
            return fordf(res, titulo=titulo)
        return res

    # --- CRUZADA ---
    if tipo == 'cruzada':
        var = variables[0]
        df_valid = data_df[data_df[var].notna()]
        tot = df_valid[factor].sum()
        # FIX 5: observed=True para evitar combinaciones fantasma con pandas moderno
        t  = df_valid.pivot_table(values=factor, index=var, columns=cruce,
                                  aggfunc='sum', fill_value=0, observed=True)
        tp = t.div(t.sum(axis=0), axis=1).mul(100).round(2)

        if var in ORDEN_CATEGORIAS:
            of = [v for v in ORDEN_CATEGORIAS[var] if v in t.index]
            tp = tp.reindex(of)
        if cruce in ORDEN_CATEGORIAS:
            oc = [v for v in ORDEN_CATEGORIAS[cruce] if v in tp.columns]
            tp = tp[oc]
        if transponer:
            tp = tp.T
        res = tp

        if estilo:
            titulo = f"Cruce: '{var}' según '{cruce}' — base ponderada: {int(tot):,} ({factor})"
            return fordf(res, titulo=titulo)
        return res

    # --- PROMEDIO / SUMA ---
    def _wavg(sub, v, f):
        d = sub[[v, f]].dropna()
        return float(round(np.average(d[v], weights=d[f]), 4)) if len(d) > 0 and d[f].sum() > 0 else np.nan

    def _wsum(sub, v, f):
        d = sub[[v, f]].dropna()
        return float(round((d[v] * d[f]).sum(), 4))

    fn       = _wavg if tipo == 'promedio' else _wsum
    col_name = 'promedio_ponderado' if tipo == 'promedio' else 'suma_ponderada'

    # Sin cruce: un valor por variable
    if not cruce:
        res = pd.DataFrame(
            [(v, fn(data_df, v, factor)) for v in variables],
            columns=['variable', col_name]
        )
        if estilo:
            titulo = f"{tipo.capitalize()} de variables — factor: {factor}"
            return fordf(res, titulo=titulo)
        return res

    # Con cruce: una fila por categoría del cruce
    filas = {
        g: {v: fn(sg, v, factor) for v in variables}
        for g, sg in data_df.groupby(cruce, observed=True)
    }
    res = pd.DataFrame(filas).T
    res.index.name = cruce

    if cruce in ORDEN_CATEGORIAS:
        ok  = [v for v in ORDEN_CATEGORIAS[cruce] if v in res.index]
        rst = [v for v in res.index if v not in ok and v != 'Total']
        # FIX 4: solo incluir 'Total' en reindex si existe en el índice
        total_fila = ['Total'] if 'Total' in res.index else []
        res = res.reindex(ok + rst + total_fila)

    if estilo:
        titulo = f"{tipo.capitalize()} cruzado por '{cruce}' — factor: {factor}"
        return fordf(res, titulo=titulo)
    return res


print('dstats: porcentajes sobre casos válidos (NaN excluidos).')

dstats: porcentajes sobre casos válidos (NaN excluidos).


## 10. Grupos de respuesta múltiple

In [13]:
_c = df.columns

GRUPOS_RM = {
    # Hogar
    'A12':  ('Pueblos indígenas o tribales (hogar)',            [c for c in _c if c.startswith('A12_') and not c.startswith('A12_1')]),
    'A13':  ('Condiciones permanentes de salud en el hogar',   [c for c in _c if c.startswith('A13_')]),
    'A14':  ('Situaciones laborales en el hogar',              [c for c in _c if c.startswith('A14_') and not c.endswith('_OTRA')]),
    # Acceso y conectividad
    'P3':   ('Dispositivos usados para acceder a internet',    [c for c in _c if c.startswith('P3_') and not c.endswith('_OTRA')]),
    'P4':   ('Formas de acceso pagado a internet en el hogar', [c for c in _c if c.startswith('P4_')]),
    'P6':   ('Razones para tener internet fijo',               [c for c in _c if c.startswith('P6_') and not c.startswith('P6_1_') and not c.endswith('_OTRA')]),
    'P6_1': ('Razones para tener internet móvil',              [c for c in _c if c.startswith('P6_1_')]),
    'P7':   ('Medidas de protección internet para menores',    [c for c in _c if c.startswith('P7_')]),
    'P8_3': ('Descripción P8_3',                               [c for c in _c if c.startswith('P8_3_')]),
    'P9':   ('Dispositivos de uso personal de menores',        [c for c in _c if c.startswith('P9_')]),
    'P12':  ('Conexión móvil 4G/5G',                           ['P12_11','P12_21','P12_31','P12_41']),
    'P13':  ('Razones de no acceso a internet fijo',           [c for c in _c if c.startswith('P13_') and not c.endswith('_OTRA')]),
    'P16':  ('Equipos que le interesaría tener (sin internet)',[c for c in _c if c.startswith('P16_')]),
    # Uso individual
    'Q6':   ('¿Cómo aprendió a usar el computador?',           [c for c in _c if c.startswith('Q6_') and c not in ['Q6_1','Q6_OTRA']]),
    'Q7_2': ('Smartphone 4G/5G',                               ['Q7_2_1','Q7_2_2','Q7_2_3','Q7_2_4']),
    'Q8':   ('Habilidades digitales',                          [c for c in _c if c.startswith('Q8_')]),
    'Q11_1':('Lugares donde usó internet ayer',                [c for c in _c if c.startswith('Q11_1_')]),
    'Q12':  ('Tipos de acceso en últimos 3 meses',             [c for c in _c if c.startswith('Q12_')]),
    'Q20':  ('Lugares donde usó internet fuera del hogar',     [c for c in _c if c.startswith('Q20_')]),
    'Q21':  ('Actividades realizadas en internet',             [c for c in _c if c.startswith('Q21_') and c not in ['Q21_1','Q21_10','Q21_19','Q21_26','Q21_33','Q21_38','Q21_44']]),
    'Q28':  ('Bienes y servicios comprados en internet',       [c for c in _c if c.startswith('Q28_') and not c.endswith('_OTRA')]),
    'Q32':  ('Actividades de seguridad y privacidad',          [c for c in _c if c.startswith('Q32_') and not c.endswith('_OTRA')]),
    'Q33':  ('Problemas de seguridad sufridos',                [c for c in _c if c.startswith('Q33_') and not c.endswith('_OTRA')]),
    'Q34':  ('Razones de no uso de internet',                  [c for c in _c if c.startswith('Q34_') and not c.endswith('_OTRA')]),
    'Q37':  ('Actividades de internet realizadas por terceros',[c for c in _c if c.startswith('Q37_')]),
    'Q39':  ('Equipos que le interesaría tener (no usuarios)', [c for c in _c if c.startswith('Q39_')]),
}

def analizar_rm(grupo, factor='fe_hogar', top_n=None, estilo=True):
    """
    Analiza grupo de respuesta múltiple — porcentajes sobre casos válidos (NaN excluidos).
    """
    if grupo not in GRUPOS_RM:
        raise ValueError(f"Grupo '{grupo}' no existe.")
    
    desc, cols = GRUPOS_RM[grupo]
    cols = [c for c in cols if c in df.columns]
    
    # construir filas con datos de cada variable
    filas = []
    for c in cols:
        # contar valores == 1 con factor ponderado
        numerador = df.loc[df[c] == 1, factor].sum()
        # contar válidos (no NaN) con factor ponderado
        denominador = df.loc[df[c].notna(), factor].sum()
        # calcular porcentaje
        porcentaje = round(numerador / denominador * 100, 1) if denominador > 0 else 0
        
        filas.append({
            'variable': c,
            'etiqueta': etiquetas_limpias.get(c, c),
            'porcentaje': porcentaje
        })
    
    # convertir a DataFrame y ordenar
    res = pd.DataFrame(filas).sort_values('porcentaje', ascending=False).reset_index(drop=True)
    
    if top_n:
        res = res.head(top_n)
    
    res.index += 1
    
    if estilo:
        base_ponderada = df.loc[df[cols].notna().any(axis=1), factor].sum()
        titulo_tabla = f"{desc} — base ponderada: {int(base_ponderada):,} ({factor})"
        return fordf(res, titulo=titulo_tabla)
    
    return res

# listar grupos disponibles
print("Grupos de respuesta múltiple disponibles:")
for k, (desc, cols) in GRUPOS_RM.items():
    cols_validas = [c for c in cols if c in df.columns]
    print(f"  '{k}': {desc} ({len(cols_validas)} opciones)")

Grupos de respuesta múltiple disponibles:
  'A12': Pueblos indígenas o tribales (hogar) (8 opciones)
  'A13': Condiciones permanentes de salud en el hogar (7 opciones)
  'A14': Situaciones laborales en el hogar (8 opciones)
  'P3': Dispositivos usados para acceder a internet (8 opciones)
  'P4': Formas de acceso pagado a internet en el hogar (5 opciones)
  'P6': Razones para tener internet fijo (13 opciones)
  'P6_1': Razones para tener internet móvil (13 opciones)
  'P7': Medidas de protección internet para menores (12 opciones)
  'P8_3': Descripción P8_3 (5 opciones)
  'P9': Dispositivos de uso personal de menores (4 opciones)
  'P12': Conexión móvil 4G/5G (4 opciones)
  'P13': Razones de no acceso a internet fijo (29 opciones)
  'P16': Equipos que le interesaría tener (sin internet) (5 opciones)
  'Q6': ¿Cómo aprendió a usar el computador? (12 opciones)
  'Q7_2': Smartphone 4G/5G (4 opciones)
  'Q8': Habilidades digitales (19 opciones)
  'Q11_1': Lugares donde usó internet ayer (9 o

In [14]:
def analizar_rm_cruce(grupo, cruce, factor='fe_personas', estilo=True):
    """
    Analiza un grupo de respuesta múltiple cruzado por una variable demográfica — solo porcentajes.
    Si estilo=True, retorna tabla estilizada (HTML). Si es False, retorna el DataFrame puro.
    """
    if grupo not in GRUPOS_RM: 
        raise ValueError(f"Grupo '{grupo}' no existe.")
        
    desc, cols = GRUPOS_RM[grupo]
    cols = [c for c in cols if c in df.columns]
    
    base_cruce = df.loc[df[cols].notna().any(axis=1)].groupby(cruce, observed=True)[factor].sum()
    
    resultados = {}
    for categoria in base_cruce.index:
        base = base_cruce[categoria]
        df_cat = df[df[cruce] == categoria]
        
        pcts = {
            etiquetas_limpias.get(c, c): round((df_cat.loc[df_cat[c]==1, factor].sum() / base) * 100, 1) 
            if base > 0 else 0 
            for c in cols
        }
        resultados[categoria] = pcts
        
    res_df = pd.DataFrame(resultados)
    res_df = res_df.sort_values(by=res_df.columns[0], ascending=False)
    
    if estilo:
        titulo_tabla = f"{desc} cruzado por '{cruce}' — factor: {factor}"
        return fordf(res_df, titulo=titulo_tabla)
    return res_df

## 11. Habilidades digitales Q8

In [15]:
# ═ DEBUG: Inspeccionar valores de Q8 ═
print("Checkeando tipos y valores de columnas Q8...")
q8_cols = [c for c in df.columns if c.startswith('Q8')]
print(f"Columnas Q8: {len(q8_cols)}")

for col in q8_cols[:3]:
    print(f"\n{col}:")
    print(f"  dtype: {df[col].dtype}")
    print(f"  primeros 5: {df[col].head(5).tolist()}")
    print(f"  únicos (primeros 5): {df[col].unique()[:5].tolist()}")
    print(f"  tipo primer valor: {type(df[col].iloc[0])}")

Checkeando tipos y valores de columnas Q8...
Columnas Q8: 19

Q8_1:
  dtype: object
  primeros 5: ['No', 'No', 'Sí', 'No', 'No']
  únicos (primeros 5): ['No', 'Sí', nan]
  tipo primer valor: <class 'str'>

Q8_2:
  dtype: object
  primeros 5: ['No', 'No', 'Sí', 'No', 'No']
  únicos (primeros 5): ['No', 'Sí', nan]
  tipo primer valor: <class 'str'>

Q8_3:
  dtype: object
  primeros 5: ['No', 'No', 'Sí', 'No', 'No']
  únicos (primeros 5): ['No', 'Sí', nan]
  tipo primer valor: <class 'str'>


### Habilidades por nivel

In [16]:
# Clasificación de habilidades Q8 por nivel de dificultad (PONDERADO)
# Valores: 'Sí', 'No', NaN

Q8_BASICO = {
    'Q8_10': 'Streaming (video/música)',
    'Q8_11': 'Juegos en línea',
    'Q8_12': 'Revisar redes sociales',
    'Q8_13': 'Publicar en redes sociales',
    'Q8_15': 'Videollamadas',
    'Q8_16': 'Correo electrónico',
}
Q8_INTERMEDIO = {
    'Q8_1':  'Procesador de texto (Word)',
    'Q8_2':  'Planilla de cálculo (Excel)',
    'Q8_3':  'Presentaciones (PowerPoint)',
    'Q8_4':  'Transferir archivos / nube',
    'Q8_5':  'Conectar nuevo dispositivo',
    'Q8_6':  'Instalar y configurar apps',
    'Q8_14': 'Editar fotos o videos',
    'Q8_17': 'Transacciones y pagos en línea',
    'Q8_18': 'Uso de IA (ChatGPT, etc.)',
}
Q8_AVANZADO = {
    'Q8_7': 'Configurar seguridad del dispositivo',
    'Q8_8': 'Instalar SO / programar (Python, Java…)',
    'Q8_9': 'Crear un sitio web',
}

_cols_b  = list(Q8_BASICO)
_cols_i  = list(Q8_INTERMEDIO)
_cols_a  = list(Q8_AVANZADO)
_cols_q8 = _cols_b + _cols_i + _cols_a + ['Q8_19']

def _nivel(row):
    """Clasifica nivel — compara con 'Sí' (no con 1)."""
    if any(row[c] == 'Sí' for c in _cols_a if pd.notna(row[c])):
        return 'Avanzado'
    if any(row[c] == 'Sí' for c in _cols_i if pd.notna(row[c])):
        return 'Intermedio'
    if any(row[c] == 'Sí' for c in _cols_b if pd.notna(row[c])):
        return 'Básico'
    return 'Sin habilidades'

# Crear nivel_habilidades
df['nivel_habilidades'] = df.apply(_nivel, axis=1)
mask_q8 = df[_cols_q8].notna().any(axis=1)

# Análisis ponderado por fe_personas
factor = 'fe_personas'
base_ponderada = df.loc[mask_q8, factor].sum()
base_ponderada_int = int(base_ponderada)

print(f"Base ponderada (fe_personas): {base_ponderada_int:,}\n")
print("Distribución por nivel (PONDERADA):")

orden_nivel = ['Avanzado', 'Intermedio', 'Básico', 'Sin habilidades']
for niv in orden_nivel:
    n_pond = df.loc[mask_q8 & (df['nivel_habilidades'] == niv), factor].sum()
    pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
    print(f"  {niv:<20} {int(n_pond):>10,}  ({pct:.1f}%)")

print("\n── Habilidades por nivel (PONDERADO) ──────────────────────────────────────────────")
for nivel, items in [('BÁSICO', Q8_BASICO), ('INTERMEDIO', Q8_INTERMEDIO), ('AVANZADO', Q8_AVANZADO)]:
    print(f"\n{nivel}")
    for cod, desc in items.items():
        n_pond = df.loc[mask_q8 & (df[cod] == 'Sí'), factor].sum()
        pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
        print(f"  {cod:<8} {desc:<45}  {int(n_pond):>10,} ({pct:.1f}%)")

Base ponderada (fe_personas): 13,286,225

Distribución por nivel (PONDERADA):
  Avanzado              5,681,644  (42.8%)
  Intermedio            5,710,998  (43.0%)
  Básico                1,585,019  (11.9%)
  Sin habilidades         308,563  (2.3%)

── Habilidades por nivel (PONDERADO) ──────────────────────────────────────────────

BÁSICO
  Q8_10    Streaming (video/música)                        7,043,538 (53.0%)
  Q8_11    Juegos en línea                                 6,068,291 (45.7%)
  Q8_12    Revisar redes sociales                         11,282,482 (84.9%)
  Q8_13    Publicar en redes sociales                      7,666,711 (57.7%)
  Q8_15    Videollamadas                                  11,271,083 (84.8%)
  Q8_16    Correo electrónico                              9,780,624 (73.6%)

INTERMEDIO
  Q8_1     Procesador de texto (Word)                      7,898,264 (59.4%)
  Q8_2     Planilla de cálculo (Excel)                     6,665,163 (50.2%)
  Q8_3     Presentaciones (Pow

In [17]:
# Tabla de habilidades por nivel — porcentajes ponderados
dstats(df, 'nivel_habilidades', tipo='frecuencia', factor='fe_personas')

,porcentaje
nivel_habilidades,
Avanzado,41.1
Básico,11.5
Intermedio,41.4
Sin habilidades,6.0


### Habilidades por tipo de uso

In [18]:
# ── Clasificación de Habilidades Digitales Q8 por tipo de uso (PONDERADO) ──────────

Q8_PRODUCTIVAS = {
    'Q8_1':  'Procesador de texto (Word)',
    'Q8_2':  'Planilla de cálculo (Excel)',
    'Q8_3':  'Presentaciones (PowerPoint)',
    'Q8_4':  'Transferir archivos / nube',
    'Q8_5':  'Conectar nuevo dispositivo',
    'Q8_6':  'Instalar y configurar apps',
    'Q8_7':  'Configurar seguridad del dispositivo',
    'Q8_8':  'Instalar SO / programar (Python, Java…)',
    'Q8_17': 'Transacciones y pagos en línea',
    'Q8_18': 'Uso de IA (ChatGPT, etc.)',
}

Q8_RECREATIVAS = {
    'Q8_10': 'Streaming (video/música)',
    'Q8_11': 'Juegos en línea',
    
}

Q8_CREATIVIDAD ={
    'Q8_14': 'Editar fotos o videos',
    'Q8_9':  'Crear un sitio web',
}

Q8_COMUNICACIONES = {
    'Q8_12': 'Revisar redes sociales',
    'Q8_13': 'Publicar en redes sociales',
    'Q8_15': 'Videollamadas',
    'Q8_16': 'Correo electrónico',
}

_cols_prod  = list(Q8_PRODUCTIVAS)
_cols_rec   = list(Q8_RECREATIVAS)
_cols_crea  = list(Q8_CREATIVIDAD)
_cols_com   = list(Q8_COMUNICACIONES)
_cols_q8_tipos = _cols_prod + _cols_rec + _cols_crea + _cols_com + ['Q8_19']

def _tipo_principal(row):
    """Clasifica tipo principal de habilidades — primero verifica productivas, luego recreativas, luego comunicaciones."""
    if any(row[c] == 'Sí' for c in _cols_prod if pd.notna(row[c])):
        return 'Productivas'
    if any(row[c] == 'Sí' for c in _cols_rec if pd.notna(row[c])):
        return 'Recreativas'
    if any(row[c] == 'Sí' for c in _cols_crea if pd.notna(row[c])):
        return 'Creativas'
    if any(row[c] == 'Sí' for c in _cols_com if pd.notna(row[c])):
        return 'Comunicaciones'
    return 'Sin habilidades'

# Crear tipo_habilidades_principal
df['tipo_habilidades_principal'] = df.apply(_tipo_principal, axis=1)
mask_q8 = df[_cols_q8_tipos].notna().any(axis=1)

# Análisis ponderado por fe_personas
factor = 'fe_personas'
base_ponderada = df.loc[mask_q8, factor].sum()
base_ponderada_int = int(base_ponderada)

print(f"Base ponderada (fe_personas): {base_ponderada_int:,}\n")
print("Distribución por tipo (PONDERADA):")

orden_tipo = ['Productivas', 'Recreativas', 'Creativas', 'Comunicaciones', 'Sin habilidades']
for tipo in orden_tipo:
    n_pond = df.loc[mask_q8 & (df['tipo_habilidades_principal'] == tipo), factor].sum()
    pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
    print(f"  {tipo:<20} {int(n_pond):>10,}  ({pct:.1f}%)")

print("\n── Habilidades por tipo (PONDERADO) ──────────────────────────────────────────────")
for tipo, items in [('PRODUCTIVAS', Q8_PRODUCTIVAS), ('RECREATIVAS', Q8_RECREATIVAS), ('CREATIVAS', Q8_CREATIVIDAD), ('COMUNICACIONES', Q8_COMUNICACIONES)]:
    print(f"\n{tipo}")
    for cod, desc in items.items():
        n_pond = df.loc[mask_q8 & (df[cod] == 'Sí'), factor].sum()
        pct = (n_pond / base_ponderada * 100) if base_ponderada > 0 else 0
        print(f"  {cod:<8} {desc:<45}  {int(n_pond):>10,} ({pct:.1f}%)")

Base ponderada (fe_personas): 13,286,225

Distribución por tipo (PONDERADA):
  Productivas          11,098,633  (83.5%)
  Recreativas             492,768  (3.7%)
  Creativas               179,815  (1.4%)
  Comunicaciones        1,206,445  (9.1%)
  Sin habilidades         308,563  (2.3%)

── Habilidades por tipo (PONDERADO) ──────────────────────────────────────────────

PRODUCTIVAS
  Q8_1     Procesador de texto (Word)                      7,898,264 (59.4%)
  Q8_2     Planilla de cálculo (Excel)                     6,665,163 (50.2%)
  Q8_3     Presentaciones (PowerPoint)                     6,910,830 (52.0%)
  Q8_4     Transferir archivos / nube                      7,115,863 (53.6%)
  Q8_5     Conectar nuevo dispositivo                      7,012,591 (52.8%)
  Q8_6     Instalar y configurar apps                      6,951,460 (52.3%)
  Q8_7     Configurar seguridad del dispositivo            5,125,336 (38.6%)
  Q8_8     Instalar SO / programar (Python, Java…)         3,053,129 (23.0%)

In [41]:
# Crear variable no_hab: 1 si todas las habilidades son 'No', 0 en otro caso
def _marca_sin_habilidades(row, cols):
    """
    Retorna 1 si todas las variables son 'No'
    Retorna 0 si al menos una es 'Sí' o hay datos pero no todos son 'No'
    Retorna np.nan si no hay datos
    """
    tiene_dato = False
    todas_son_no = True
    
    for col in cols:
        if col in row.index and pd.notna(row[col]):
            tiene_dato = True
            if row[col] != 'No':  # Si alguna NO es 'No'
                todas_son_no = False
    
    if not tiene_dato:
        return np.nan  # Sin datos, devuelve NaN
    
    if todas_son_no:
        return 1  # Todas son 'No'
    else:
        return 0  # Al menos una no es 'No'

# Aplicar la función usando todas las columnas de habilidades
todas_cols_habilidades = (list(Q8_PRODUCTIVAS) + list(Q8_RECREATIVAS) + 
                          list(Q8_CREATIVIDAD) + list(Q8_COMUNICACIONES) + 
                          list(Q8_AVANZADO) + list(Q8_INTERMEDIO) + 
                          list(Q8_BASICO))

df['no_hab'] = df.apply(lambda row: _marca_sin_habilidades(row, todas_cols_habilidades), axis=1)

print('Variable creada: no_hab')
print(f'Casos con no_hab=1 (sin habilidades): {(df["no_hab"] == 1).sum()}')
print(f'Casos con no_hab=0 (con al menos una habilidad): {(df["no_hab"] == 0).sum()}')

df['no_hab'] = df['no_hab'].map({
    0: 'Al menos una habilidad',
    1: 'Sin habilidades',
    np.nan: np.nan  # Mantiene NaN como NaN
})

Variable creada: no_hab
Casos con no_hab=1 (sin habilidades): 179
Casos con no_hab=0 (con al menos una habilidad): 4579


# Análisis

## Caracterización Muestral Personas

In [19]:
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'region', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'zona', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'educ', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'ocupacion_encuestado', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'tramo_edad', tipo='frecuencia', factor='fe_personas'))
display(dstats(df, 'gse', tipo='frecuencia', factor='fe_personas'))


,porcentaje
sexo,
Hombre,48.4
Mujer,51.6


,porcentaje
region,
Tarapacá,1.8
Antofagasta,3.4
Atacama,1.6
Coquimbo,4.2
Valparaíso,10.5
O'Higgins,5.2
Maule,5.9
Biobío,8.8
Araucanía,5.4


,porcentaje
zona,
Urbana,87.6
Rural,12.4


,porcentaje
educ,
Sin educación formal,0.3
Básica incompleta,4.3
Básica completa,6.5
Media CH incompleta,6.5
Media TP incompleta,3.4
Media CH completa,23.4
Media TP completa,9.0
Superior técnica incompleta,3.4
Superior técnica completa,11.2


,porcentaje
ocupacion_encuestado,
Trabajos ocasionales e informales,9.3
Oficio menor - obrero no calificado,7.0
Obrero calificado - microempresario,23.8
Empleado medio - técnico - prof. independiente,30.6
Ejecutivo medio - prof. universitario,9.4
Alto ejecutivo - empresario - directivo,1.0
Sin trabajo remunerado,18.9


,porcentaje
sexo,
Hombre,48.4
Mujer,51.6


,porcentaje
tramo_edad,
Menor de 18,1.2
18-29,25.8
30-44,29.9
45-59,21.1
60 y más,22.1


,porcentaje
gse,
AB,11.0
C1,13.4
C2,23.2
C3,23.5
D,15.2
E,13.7


## Caracterización muestral hogares

In [20]:
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'region', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'zona', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'educ', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'ocupacion_encuestado', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'sexo', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'tramo_edad', tipo='frecuencia', factor='fe_hogar'))
display(dstats(df, 'gse', tipo='frecuencia', factor='fe_hogar'))


,porcentaje
sexo,
Hombre,45.2
Mujer,54.8


,porcentaje
region,
Tarapacá,1.7
Antofagasta,3.1
Atacama,1.6
Coquimbo,4.2
Valparaíso,10.8
O'Higgins,5.3
Maule,6.2
Biobío,9.0
Araucanía,5.6


,porcentaje
zona,
Urbana,87.6
Rural,12.4


,porcentaje
educ,
Sin educación formal,0.5
Básica incompleta,5.9
Básica completa,7.8
Media CH incompleta,7.3
Media TP incompleta,3.3
Media CH completa,23.8
Media TP completa,9.2
Superior técnica incompleta,2.9
Superior técnica completa,11.2


,porcentaje
ocupacion_encuestado,
Trabajos ocasionales e informales,10.1
Oficio menor - obrero no calificado,7.9
Obrero calificado - microempresario,25.2
Empleado medio - técnico - prof. independiente,30.1
Ejecutivo medio - prof. universitario,8.8
Alto ejecutivo - empresario - directivo,1.0
Sin trabajo remunerado,16.9


,porcentaje
sexo,
Hombre,45.2
Mujer,54.8


,porcentaje
tramo_edad,
Menor de 18,0.8
18-29,17.8
30-44,26.4
45-59,23.3
60 y más,31.8


,porcentaje
gse,
AB,10.0
C1,12.6
C2,22.4
C3,24.0
D,15.8
E,15.2


## Conectividad e infra (Hogares)

### Nacional (sin cruces)

In [21]:
display(dstats(df, 'acceso_internet_hogar', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='frecuencia', factor='fe_hogar', estilo=True))
display(analizar_rm('P3', factor='fe_hogar', estilo=True))
display(analizar_rm('P4', factor='fe_hogar', estilo=True))
display(analizar_rm('P6', factor='fe_hogar', estilo=True))
display(analizar_rm('P6_1', factor='fe_hogar', estilo=True))
display(analizar_rm('P7', factor='fe_hogar', estilo=True))
display(analizar_rm('P9', factor='fe_hogar', estilo=True))
display(analizar_rm('P8_3', factor='fe_hogar', estilo=True))


,porcentaje
acceso_internet_hogar,
Sí,96.6
No,3.4


,porcentaje
n_computadores_hogar,
0.000000,30.9
1.000000,37.4
2.000000,20.3
3.000000,7.6
4.000000,2.2
5.000000,0.9
6.000000,0.3
7.000000,0.1
8.000000,0.2


,porcentaje
n_smartphones_hogar,
0.000000,0.3
1.000000,18.6
2.000000,29.1
3.000000,23.7
4.000000,16.9
5.000000,6.0
6.000000,2.7
7.000000,1.5
8.000000,0.4


,variable,promedio_ponderado
0,n_computadores_hogar,1.2


,variable,promedio_ponderado
0,n_smartphones_hogar,2.9


...

,variable,etiqueta,porcentaje
1,P6_3,Permite comunicarse con otras personas,65.1
2,P6_2,Permite tener más acceso a información,64.6
3,P6_6,"Permite acceder a entretenimiento a través de streaming de películas, series y música",56.3
4,P6_8,"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",54.4
5,P6_1,Apoyo a la educación propia o de hijos / nietos / parientes,51.8
6,P6_12,Permite realizar trabajo o estudios desde casa,48.9
7,P6_4,Permite participar en redes sociales y comunidades en línea,48.4
8,P6_11,Proporciona acceso a noticias y actualidad,46.0
9,P6_10,Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,42.4
10,P6_5,Permite acceder a juegos y otros medios de entretención,37.4


,variable,etiqueta,porcentaje
1,P6_1_3,Permite comunicarse con otras personas,82.1
2,P6_1_2,Permite tener más acceso a información,53.8
3,P6_1_4,Permite participar en redes sociales y comunidades en línea,46.7
4,P6_1_8,"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",43.3
5,P6_1_6,"Permite acceder a entretenimiento a través de streaming de películas, series y música",37.9
6,P6_1_11,Proporciona acceso a noticias y actualidad,34.6
7,P6_1_12,Permite realizar trabajo o estudios desde casa,32.2
8,P6_1_10,Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,30.7
9,P6_1_1,Apoyo a la educación propia o de hijos / nietos / parientes,30.1
10,P6_1_5,Permite acceder a juegos y otros medios de entretención,28.7


,variable,etiqueta,porcentaje
1,P7_1,"Se acuerdan reglas sobre el uso de Internet (horarios, tiempo y condiciones)",49.8
2,P7_3,"Supervisión y/o monitoreo de uso de Internet (en presencia de adulto, verificar historial de navegación, etc.)",41.3
3,P7_5,"Educar a los niños sobre el uso seguro y responsable de Internet (ej: no publicar sus nombres completos en redes sociales, qué se puede hacer, cómo detectar relaciones / interacciones inseguras, etc.)",36.1
4,P7_7,Restricción o bloqueo de ciertas aplicaciones o plataformas en línea.,25.5
5,P7_2,Instalación de filtros de Internet (software de control parental),22.9
6,P7_10,Uso de controles parentales en los dispositivos conectados a internet.,22.5
7,P7_11,No es necesario tener medidas de protección,22.2
8,P7_9,Restricción de descargas o instalación de aplicaciones sin permiso.,22.1
9,P7_6,Filtro de contenido para limitar el acceso a sitios web inapropiados.,21.3
10,P7_8,Restricción de acceso a redes sociales y mensajería instantánea.,19.5


,variable,etiqueta,porcentaje
1,P9_1,Smartphone,63.8
2,P9_2,Computador / Notebook,38.6
3,P9_4,"Consola de juegos (WII, Nintendo, PS-3, PS-4, PS-5, XBOX,etc.)",27.2
4,P9_3,"Tablet (Ipad, Galaxy TAB, Xperia Tablet, HP palm touchpad, etc.)",22.8


,variable,etiqueta,porcentaje
1,P8_3_4,No he podido detectar ninguna situación,74.8
2,P8_3_3,Intentos de engaños estafas o robo de información personal,16.0
3,P8_3_1,"Ser molestado, recibir hostilidad u hostigamiento por Internet (cyber bullying)",7.9
4,P8_3_5,NS/NR,5.4
5,P8_3_2,"Intentos de captación o seducción, relaciones inapropiadas con menores de edad",3.4


### POR GSE

In [22]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='gse', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='gse', factor='fe_hogar', estilo=True))

gse,AB,C1,C2,C3,D,E
acceso_internet_hogar,,,,,,
Sí,99.8,99.2,98.9,96.1,95.3,90.8
No,0.2,0.8,1.1,3.9,4.7,9.2


gse,AB,C1,C2,C3,D,E
n_computadores_hogar,,,,,,
0.000000,5.4,12.8,25.2,37.7,44.9,48.5
1.000000,28.3,36.1,41.8,42.8,32.4,34.3
2.000000,39.7,30.7,20.7,14.0,15.3,12.2
3.000000,15.9,12.6,8.7,4.2,4.5,4.6
4.000000,8.1,4.8,1.3,1.0,1.2,0.2
5.000000,2.3,1.2,1.7,0.2,0.1,0.1
6.000000,0.0,1.0,0.1,0.1,0.7,0.0
7.000000,0.0,0.1,0.5,0.0,0.1,0.0
8.000000,0.1,0.8,0.1,0.0,0.7,0.0


gse,AB,C1,C2,C3,D,E
n_smartphones_hogar,,,,,,
0.000000,0.0,0.2,0.4,0.2,0.3,0.5
1.000000,18.3,18.7,18.6,16.8,17.6,22.7
2.000000,34.1,26.1,30.4,27.8,27.6,29.8
3.000000,19.5,29.1,22.0,26.8,24.0,19.5
4.000000,20.9,18.1,16.6,17.1,15.7,14.1
5.000000,3.4,4.1,7.3,7.0,6.6,5.5
6.000000,2.6,2.7,2.0,2.3,3.7,3.4
7.000000,1.0,0.1,1.8,1.0,2.6,2.1
8.000000,0.1,0.8,0.2,0.1,0.3,1.2


,n_computadores_hogar
gse,
AB,2.0
C1,1.7
C2,1.3
C3,0.9
D,1.0
E,0.9


,n_smartphones_hogar
gse,
AB,2.7
C1,2.8
C2,2.8
C3,3.0
D,3.0
E,2.8


...

,AB,C1,C2,C3,D,E
Permite tener más acceso a información,77.3,69.9,70.9,60.6,51.1,55.7
Permite realizar trabajo o estudios desde casa,74.8,59.2,54.7,38.4,36.4,31.6
Permite comunicarse con otras personas,74.6,64.2,68.7,66.5,54.9,58.5
"Permite acceder a entretenimiento a través de streaming de películas, series y música",73.0,62.8,58.9,52.6,46.4,44.6
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",71.6,59.2,61.8,50.1,39.7,41.1
Permite participar en redes sociales y comunidades en línea,65.5,50.7,57.2,44.9,32.9,34.5
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,63.8,50.9,51.2,34.4,26.1,26.1
Proporciona acceso a noticias y actualidad,62.7,50.2,54.0,39.9,36.7,29.4
Permite acceder a servicios de telemedicina o consultas médicas en línea,57.5,36.8,42.2,30.6,24.8,26.5
Permite acceder a juegos y otros medios de entretención,51.6,39.2,41.7,33.4,28.3,29.9


,AB,C1,C2,C3,D,E
Permite comunicarse con otras personas,88.4,80.2,79.0,85.0,81.0,80.6
Permite tener más acceso a información,63.8,53.0,58.3,52.7,47.2,48.7
Permite participar en redes sociales y comunidades en línea,61.1,45.5,52.4,46.3,37.9,37.4
Permite realizar trabajo o estudios desde casa,55.5,37.3,37.5,23.3,26.9,22.3
Proporciona acceso a noticias y actualidad,54.5,42.1,40.1,29.3,24.0,24.3
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",54.2,48.2,50.2,41.4,35.6,30.7
"Permite acceder a entretenimiento a través de streaming de películas, series y música",52.7,41.7,43.9,32.6,32.7,27.8
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,45.9,33.9,40.8,24.8,20.8,20.2
Permite acceder a juegos y otros medios de entretención,41.2,31.0,35.1,26.4,20.1,20.1
Permite acceder a servicios de telemedicina o consultas médicas en línea,38.2,28.2,26.7,19.6,16.7,18.1


,AB,C1,C2,C3,D,E
"Supervisión y/o monitoreo de uso de Internet (en presencia de adulto, verificar historial de navegación, etc.)",59.8,57.5,37.8,36.2,38.2,32.2
"Se acuerdan reglas sobre el uso de Internet (horarios, tiempo y condiciones)",56.1,57.7,47.6,51.4,51.2,37.9
"Educar a los niños sobre el uso seguro y responsable de Internet (ej: no publicar sus nombres completos en redes sociales, qué se puede hacer, cómo detectar relaciones / interacciones inseguras, etc.)",44.6,49.9,34.8,30.9,34.4,31.8
Instalación de filtros de Internet (software de control parental),33.4,31.7,23.4,20.0,23.8,11.4
Restricción de descargas o instalación de aplicaciones sin permiso.,29.8,33.8,21.1,19.2,20.6,15.3
Restricción o bloqueo de ciertas aplicaciones o plataformas en línea.,28.5,29.1,23.0,25.8,27.6,21.8
Restricción de acceso a redes sociales y mensajería instantánea.,27.0,28.9,19.0,13.0,22.8,15.5
Uso de controles parentales en los dispositivos conectados a internet.,26.3,35.3,21.5,22.1,17.3,16.8
No es necesario tener medidas de protección,22.7,16.8,28.2,22.7,20.1,17.2
Filtro de contenido para limitar el acceso a sitios web inapropiados.,22.6,26.5,21.2,22.0,20.1,16.5


,AB,C1,C2,C3,D,E
Smartphone,63.8,53.0,59.8,66.4,65.2,72.7
Computador / Notebook,51.6,31.5,38.7,35.4,37.9,40.3
"Consola de juegos (WII, Nintendo, PS-3, PS-4, PS-5, XBOX,etc.)",46.7,21.7,28.3,23.2,20.3,29.1
"Tablet (Ipad, Galaxy TAB, Xperia Tablet, HP palm touchpad, etc.)",33.2,23.7,25.7,19.4,20.9,17.5


,AB,C1,C2,C3,D,E
No he podido detectar ninguna situación,72.9,70.4,69.6,79.6,78.7,75.7
Intentos de engaños estafas o robo de información personal,17.9,20.9,20.4,14.5,12.7,9.4
NS/NR,5.4,6.0,7.1,2.9,3.8,8.1
"Ser molestado, recibir hostilidad u hostigamiento por Internet (cyber bullying)",4.0,7.0,11.7,4.9,7.7,10.5
"Intentos de captación o seducción, relaciones inapropiadas con menores de edad",2.8,0.8,5.7,3.7,2.5,2.0


### POR REGION

In [23]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='region', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='region', factor='fe_hogar', estilo=True))

region,Tarapacá,Antofagasta,Atacama,Coquimbo,Valparaíso,O'Higgins,Maule,Biobío,Araucanía,Los Lagos,Aysén,Magallanes,Metropolitana,Los Ríos,Arica y Parinacota,Ñuble
acceso_internet_hogar,,,,,,,,,,,,,,,,
Sí,98.7,97.2,99.0,98.2,97.0,97.5,97.2,93.2,96.4,96.0,99.2,96.6,96.7,96.9,94.1,96.1
No,1.3,2.8,1.0,1.8,3.0,2.5,2.8,6.8,3.6,4.0,0.8,3.4,3.3,3.1,5.9,3.9


region,Tarapacá,Antofagasta,Atacama,Coquimbo,Valparaíso,O'Higgins,Maule,Biobío,Araucanía,Los Lagos,Aysén,Magallanes,Metropolitana,Los Ríos,Arica y Parinacota,Ñuble
n_computadores_hogar,,,,,,,,,,,,,,,,
0.000000,35.9,33.2,29.8,30.8,34.4,35.6,36.6,28.1,27.4,36.9,20.9,27.5,26.7,43.1,35.4,47.6
1.000000,41.9,35.3,35.4,38.4,40.9,40.1,33.8,45.1,37.3,39.2,47.5,40.0,34.4,42.9,33.7,34.5
2.000000,17.3,20.9,22.4,19.3,15.4,16.3,21.5,16.4,21.0,13.4,20.9,17.9,24.9,10.8,18.6,11.8
3.000000,3.5,7.7,7.8,8.0,6.8,6.3,6.3,6.9,10.5,7.1,7.2,9.7,8.8,2.1,6.2,4.2
4.000000,0.7,2.5,0.0,2.2,2.3,0.9,1.2,2.4,2.5,2.5,2.6,4.2,2.6,1.1,3.5,1.1
5.000000,0.0,0.4,0.7,0.6,0.2,0.8,0.6,0.8,0.9,0.6,0.8,0.7,1.3,0.0,1.7,0.4
6.000000,0.7,0.0,1.4,0.0,0.0,0.0,0.0,0.0,0.3,0.2,0.0,0.0,0.5,0.0,0.0,0.0
7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0,0.2,0.0,0.0,0.3,0.0,0.9,0.0
8.000000,0.0,0.0,0.0,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0


region,Tarapacá,Antofagasta,Atacama,Coquimbo,Valparaíso,O'Higgins,Maule,Biobío,Araucanía,Los Lagos,Aysén,Magallanes,Metropolitana,Los Ríos,Arica y Parinacota,Ñuble
n_smartphones_hogar,,,,,,,,,,,,,,,,
0.000000,0.0,0.5,0.0,0.3,0.5,0.0,0.3,0.3,0.2,0.0,0.0,0.0,0.3,0.5,0.9,0.8
1.000000,21.8,17.4,14.7,17.0,31.1,22.3,14.9,20.2,13.0,15.5,33.7,17.2,16.1,16.5,22.8,19.9
2.000000,30.4,21.3,30.0,26.3,28.6,32.1,34.6,34.7,30.0,35.7,36.1,38.0,25.3,27.8,27.7,42.4
3.000000,23.8,19.9,24.5,23.4,20.9,20.9,24.5,24.1,27.6,25.6,20.9,26.2,24.6,24.8,19.5,19.5
4.000000,13.9,19.8,15.0,20.4,12.2,15.1,16.4,13.4,17.9,14.2,7.7,10.3,19.4,21.1,15.9,12.5
5.000000,6.0,11.7,9.9,6.1,4.1,6.0,6.2,5.1,6.9,6.0,0.8,6.9,6.3,6.7,6.2,2.1
6.000000,2.1,6.2,4.0,2.9,1.9,1.5,1.8,0.8,3.8,2.3,0.8,0.0,3.4,1.5,4.4,1.0
7.000000,2.1,1.8,0.0,1.6,0.5,1.1,0.3,0.5,0.3,0.5,0.0,0.7,2.6,1.0,0.9,0.8
8.000000,0.0,1.1,0.0,0.9,0.0,0.5,0.3,0.0,0.0,0.3,0.0,0.0,0.5,0.0,0.9,1.1


,n_computadores_hogar
region,
Tarapacá,0.9
Antofagasta,1.1
Atacama,3.4
Coquimbo,1.2
Valparaíso,1.0
O'Higgins,1.0
Maule,1.0
Biobío,1.1
Araucanía,1.3


,n_smartphones_hogar
region,
Tarapacá,2.7
Antofagasta,3.2
Atacama,3.3
Coquimbo,3.0
Valparaíso,2.4
O'Higgins,2.9
Maule,2.8
Biobío,2.6
Araucanía,2.9


...

,Antofagasta,Araucanía,Arica y Parinacota,Atacama,Aysén,Biobío,Coquimbo,Los Lagos,Los Ríos,Magallanes,Maule,Metropolitana,O'Higgins,Tarapacá,Valparaíso,Ñuble
Permite tener más acceso a información,70.0,53.4,55.8,68.3,62.9,68.7,50.7,58.1,62.4,62.9,65.1,65.9,59.3,72.5,70.0,68.6
Apoyo a la educación propia o de hijos / nietos / parientes,54.5,51.9,46.3,59.5,49.0,50.5,51.7,51.6,38.9,41.4,57.7,52.5,57.0,56.0,44.6,59.5
"Permite acceder a entretenimiento a través de streaming de películas, series y música",52.5,47.4,45.1,55.5,51.0,42.4,49.9,43.5,53.0,73.3,63.8,58.3,55.5,61.8,66.5,70.5
Permite comunicarse con otras personas,47.4,53.2,54.5,78.9,58.0,67.6,50.0,62.6,47.6,68.9,63.3,67.6,55.5,74.7,79.6,60.2
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",44.3,43.1,48.6,59.8,48.6,41.7,44.1,49.6,13.6,61.2,54.7,61.4,47.8,53.9,60.7,55.8
Proporciona acceso a noticias y actualidad,39.2,37.3,40.2,59.4,42.5,38.9,34.1,41.4,5.0,60.4,49.2,50.8,40.3,25.3,56.1,37.7
Permite participar en redes sociales y comunidades en línea,36.9,29.9,47.5,56.1,43.3,52.5,37.8,42.8,20.6,49.1,44.6,53.6,37.4,52.6,55.3,41.2
Permite acceder a juegos y otros medios de entretención,34.1,27.5,39.2,36.0,37.1,37.9,31.6,30.4,9.2,39.6,35.9,42.7,26.0,36.5,39.5,28.3
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,30.8,31.4,28.4,51.5,36.3,29.2,27.8,35.3,6.4,49.1,41.6,51.8,34.9,27.5,45.6,35.0
Permite realizar trabajo o estudios desde casa,30.2,44.0,39.1,46.3,48.2,35.4,43.8,41.5,12.1,53.5,58.1,56.2,43.2,45.7,49.3,42.0


,Antofagasta,Araucanía,Arica y Parinacota,Atacama,Aysén,Biobío,Coquimbo,Los Lagos,Los Ríos,Magallanes,Maule,Metropolitana,O'Higgins,Tarapacá,Valparaíso,Ñuble
Permite comunicarse con otras personas,80.2,73.0,75.1,79.1,70.0,79.4,74.7,74.4,90.8,79.6,86.6,82.2,86.4,65.1,89.8,90.5
Permite tener más acceso a información,61.4,48.7,43.4,55.0,52.1,51.7,46.2,60.7,46.4,60.7,57.0,50.6,59.1,53.5,63.2,59.6
Permite participar en redes sociales y comunidades en línea,53.9,36.0,40.6,51.2,43.1,43.1,42.4,47.3,50.3,56.9,51.2,46.7,41.2,50.4,50.0,51.5
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",48.6,38.9,38.7,53.7,38.6,30.3,37.7,45.9,10.3,58.4,51.9,44.1,38.5,42.7,50.6,53.3
"Permite acceder a entretenimiento a través de streaming de películas, series y música",46.7,31.7,36.1,37.6,32.8,26.5,28.8,35.8,29.7,55.4,39.3,38.9,31.9,34.7,47.8,45.0
Permite acceder a juegos y otros medios de entretención,43.4,20.8,26.1,31.1,26.2,25.6,18.4,27.4,10.2,37.8,31.8,31.2,21.4,29.0,32.7,22.4
Proporciona acceso a noticias y actualidad,39.1,29.6,33.2,49.7,30.7,30.6,23.5,37.0,2.2,49.3,41.8,35.3,32.3,18.5,43.5,30.7
Apoyo a la educación propia o de hijos / nietos / parientes,38.6,23.3,33.3,50.8,37.2,27.8,25.0,39.2,13.8,29.6,32.5,30.6,30.8,36.9,27.0,25.9
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,36.6,26.2,23.2,41.3,25.5,20.3,21.9,30.2,3.8,46.2,35.2,33.2,29.6,21.3,38.4,18.4
Permite acceder a servicios de telemedicina o consultas médicas en línea,31.5,18.9,15.8,33.3,18.6,14.7,18.2,19.6,1.6,28.8,27.9,26.6,23.0,12.8,29.2,11.3


,Antofagasta,Araucanía,Arica y Parinacota,Atacama,Aysén,Biobío,Coquimbo,Los Lagos,Los Ríos,Magallanes,Maule,Metropolitana,O'Higgins,Tarapacá,Valparaíso,Ñuble
"Se acuerdan reglas sobre el uso de Internet (horarios, tiempo y condiciones)",50.0,56.1,36.2,48.3,60.2,57.5,56.5,64.1,2.9,75.6,52.2,45.0,53.1,66.2,53.1,50.9
"Supervisión y/o monitoreo de uso de Internet (en presencia de adulto, verificar historial de navegación, etc.)",47.3,42.4,39.0,48.3,53.8,34.4,39.5,49.0,52.2,69.0,47.4,40.3,39.4,54.2,33.5,24.1
"Educar a los niños sobre el uso seguro y responsable de Internet (ej: no publicar sus nombres completos en redes sociales, qué se puede hacer, cómo detectar relaciones / interacciones inseguras, etc.)",37.4,33.8,33.4,41.8,50.5,28.0,46.5,39.2,2.9,62.4,45.4,37.3,33.1,40.8,32.3,26.9
Restricción o bloqueo de ciertas aplicaciones o plataformas en línea.,31.1,32.8,27.8,23.4,43.0,17.7,30.8,29.3,1.4,35.7,30.1,26.5,24.6,27.3,18.8,19.4
"Permitir acceso a Internet sólo desde lugares comunes / públicas de la casa (living, comedor, etc.)",28.8,16.5,19.5,18.4,44.1,12.7,25.7,21.6,0.0,40.1,24.8,15.7,20.0,9.6,16.0,17.3
Restricción de acceso a redes sociales y mensajería instantánea.,27.6,26.5,16.7,15.6,35.5,8.4,20.2,17.1,0.0,37.8,23.9,22.6,14.9,14.6,16.4,9.9
Restricción de descargas o instalación de aplicaciones sin permiso.,23.2,29.1,22.3,23.4,46.2,13.1,30.0,23.8,0.0,37.9,31.5,21.9,17.1,15.8,26.0,4.3
Filtro de contenido para limitar el acceso a sitios web inapropiados.,22.0,28.5,13.9,22.6,37.6,12.8,28.1,23.8,1.4,42.4,24.8,21.6,22.9,12.0,22.4,11.8
Instalación de filtros de Internet (software de control parental),21.2,26.9,16.7,22.6,50.5,27.6,26.2,24.8,11.3,26.8,20.0,22.4,20.6,18.9,27.5,9.6
Uso de controles parentales en los dispositivos conectados a internet.,19.7,20.3,13.9,19.8,36.6,16.7,29.9,22.5,1.4,40.1,21.1,24.6,22.3,13.9,29.9,7.4


,Antofagasta,Araucanía,Arica y Parinacota,Atacama,Aysén,Biobío,Coquimbo,Los Lagos,Los Ríos,Magallanes,Maule,Metropolitana,O'Higgins,Tarapacá,Valparaíso,Ñuble
Smartphone,66.9,62.1,58.2,59.7,48.4,68.9,71.9,67.3,47.7,55.5,59.7,65.6,57.1,68.8,59.4,58.3
"Consola de juegos (WII, Nintendo, PS-3, PS-4, PS-5, XBOX,etc.)",34.2,21.3,33.4,29.1,18.3,27.0,29.8,16.5,11.3,35.5,15.8,32.3,18.9,33.8,29.1,19.2
Computador / Notebook,30.7,39.4,49.9,32.5,34.4,50.0,44.7,40.9,22.5,48.8,35.9,38.4,38.9,35.0,35.9,29.0
"Tablet (Ipad, Galaxy TAB, Xperia Tablet, HP palm touchpad, etc.)",27.2,22.9,25.1,22.0,32.3,28.7,24.5,17.2,9.9,46.7,19.6,21.7,26.9,16.5,29.1,14.5


,Antofagasta,Araucanía,Arica y Parinacota,Atacama,Aysén,Biobío,Coquimbo,Los Lagos,Los Ríos,Magallanes,Maule,Metropolitana,O'Higgins,Tarapacá,Valparaíso,Ñuble
No he podido detectar ninguna situación,64.2,59.4,72.2,66.0,45.2,79.8,73.0,74.8,97.2,75.5,73.8,74.5,82.9,65.0,79.2,85.8
NS/NR,14.6,10.8,8.4,4.2,24.7,5.7,2.6,5.3,2.8,6.7,5.3,5.0,4.6,3.8,2.4,2.8
Intentos de engaños estafas o robo de información personal,12.2,22.1,19.5,24.1,22.6,13.0,16.6,14.0,0.0,11.2,14.3,18.6,9.1,24.2,13.6,11.4
"Ser molestado, recibir hostilidad u hostigamiento por Internet (cyber bullying)",7.5,13.9,5.6,5.7,14.0,3.6,7.9,5.9,0.0,8.9,9.0,9.4,5.7,7.7,6.0,6.2
"Intentos de captación o seducción, relaciones inapropiadas con menores de edad",6.3,5.6,0.0,5.0,3.2,3.2,6.1,4.3,0.0,2.2,4.8,2.0,1.7,5.0,6.0,2.8


### POR ZONA

In [24]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='zona', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='zona', factor='fe_hogar', estilo=True))

zona,Urbana,Rural
acceso_internet_hogar,,
Sí,96.8,95.1
No,3.2,4.9


zona,Urbana,Rural
n_computadores_hogar,,
0.000000,29.5,41.6
1.000000,37.5,36.7
2.000000,21.0,15.3
3.000000,8.1,4.5
4.000000,2.4,1.2
5.000000,0.9,0.3
6.000000,0.3,0.1
7.000000,0.1,0.1
8.000000,0.2,0.1


zona,Urbana,Rural
n_smartphones_hogar,,
0.000000,0.3,0.2
1.000000,19.1,14.7
2.000000,28.6,32.4
3.000000,23.4,25.9
4.000000,16.8,17.3
5.000000,6.0,6.0
6.000000,2.7,2.6
7.000000,1.6,0.3
8.000000,0.4,0.1


,n_computadores_hogar
zona,
Urbana,1.3
Rural,1.0


,n_smartphones_hogar
zona,
Urbana,2.9
Rural,2.8


...

,Rural,Urbana
Apoyo a la educación propia o de hijos / nietos / parientes,63.1,50.8
Permite comunicarse con otras personas,62.1,65.4
Permite tener más acceso a información,60.0,65.0
"Permite acceder a entretenimiento a través de streaming de películas, series y música",54.9,56.5
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",47.1,55.1
Permite realizar trabajo o estudios desde casa,45.9,49.1
Permite participar en redes sociales y comunidades en línea,41.7,48.9
Proporciona acceso a noticias y actualidad,38.3,46.7
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,33.6,43.2
Permite acceder a juegos y otros medios de entretención,33.0,37.8


,Rural,Urbana
Permite comunicarse con otras personas,87.0,81.4
Permite tener más acceso a información,56.6,53.4
Permite participar en redes sociales y comunidades en línea,48.0,46.5
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",40.6,43.7
"Permite acceder a entretenimiento a través de streaming de películas, series y música",35.6,38.2
Proporciona acceso a noticias y actualidad,34.1,34.7
Apoyo a la educación propia o de hijos / nietos / parientes,32.7,29.7
Permite realizar trabajo o estudios desde casa,28.9,32.7
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,23.8,31.7
Permite acceder a juegos y otros medios de entretención,23.6,29.5


,Rural,Urbana
"Se acuerdan reglas sobre el uso de Internet (horarios, tiempo y condiciones)",48.9,49.9
"Supervisión y/o monitoreo de uso de Internet (en presencia de adulto, verificar historial de navegación, etc.)",39.7,41.6
"Educar a los niños sobre el uso seguro y responsable de Internet (ej: no publicar sus nombres completos en redes sociales, qué se puede hacer, cómo detectar relaciones / interacciones inseguras, etc.)",28.1,37.6
No es necesario tener medidas de protección,26.9,21.3
Restricción o bloqueo de ciertas aplicaciones o plataformas en línea.,23.9,25.8
Restricción de descargas o instalación de aplicaciones sin permiso.,20.7,22.4
Uso de controles parentales en los dispositivos conectados a internet.,19.0,23.1
Filtro de contenido para limitar el acceso a sitios web inapropiados.,17.2,22.1
Instalación de filtros de Internet (software de control parental),16.9,24.1
Restricción de acceso a redes sociales y mensajería instantánea.,16.2,20.1


,Rural,Urbana
Smartphone,61.1,64.3
Computador / Notebook,36.3,39.0
"Tablet (Ipad, Galaxy TAB, Xperia Tablet, HP palm touchpad, etc.)",18.7,23.7
"Consola de juegos (WII, Nintendo, PS-3, PS-4, PS-5, XBOX,etc.)",18.4,28.9


,Rural,Urbana
No he podido detectar ninguna situación,79.7,73.8
Intentos de engaños estafas o robo de información personal,11.7,16.9
"Ser molestado, recibir hostilidad u hostigamiento por Internet (cyber bullying)",6.3,8.2
NS/NR,4.1,5.6
"Intentos de captación o seducción, relaciones inapropiadas con menores de edad",2.8,3.5


### POR EDUCACIÓN JEFE DE HOGAR

In [33]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='educ_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='educ_jh', factor='fe_hogar', estilo=True))

educ_jh,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
acceso_internet_hogar,,,,,,,,,,,
Sí,93.2,88.6,97.7,91.2,97.7,98.5,80.6,99.5,99.1,99.5,97.8
No,6.8,11.4,2.3,8.8,2.3,1.5,19.4,0.5,0.9,0.5,2.2


educ_jh,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
n_computadores_hogar,,,,,,,,,,,
0.000000,51.3,52.6,36.0,48.1,30.9,33.8,58.5,22.3,21.0,10.6,12.5
1.000000,28.8,31.9,44.2,33.0,42.2,35.1,35.6,39.8,36.7,30.4,47.9
2.000000,14.8,9.8,15.4,16.3,15.3,20.4,5.8,21.1,17.3,36.4,22.7
3.000000,3.5,3.9,3.4,2.0,8.2,9.3,0.0,13.0,13.1,13.8,9.4
4.000000,1.6,0.2,0.7,0.4,1.7,0.9,0.0,2.4,0.8,6.7,0.8
5.000000,0.0,0.0,0.1,0.0,1.6,0.5,0.0,1.5,6.0,1.2,2.3
6.000000,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.6,2.0
7.000000,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,5.0,0.0,0.4
8.000000,0.0,1.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,1.9


educ_jh,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
n_smartphones_hogar,,,,,,,,,,,
0.000000,0.4,0.8,0.6,0.0,0.0,0.3,1.6,0.0,0.0,0.1,0.0
1.000000,18.9,21.5,15.9,14.3,19.1,16.6,17.6,22.9,15.0,17.9,27.5
2.000000,29.6,25.1,33.1,25.1,25.8,20.1,23.6,27.3,31.1,33.5,18.2
3.000000,22.1,20.5,25.4,22.3,27.0,19.1,34.0,22.6,16.4,24.2,24.7
4.000000,12.5,12.3,15.5,21.5,16.8,27.2,4.1,18.4,26.2,18.8,13.1
5.000000,6.6,8.5,4.7,12.9,6.2,5.9,16.3,6.0,8.0,3.7,6.1
6.000000,5.5,1.4,2.7,0.9,2.2,4.5,2.7,2.3,1.9,1.5,6.0
7.000000,1.8,4.5,1.7,2.7,1.8,0.0,0.0,0.1,1.4,0.1,2.4
8.000000,0.3,2.0,0.0,0.4,0.4,0.3,0.0,0.1,0.0,0.1,1.9


,n_computadores_hogar
educ_jh,
Básica completa,0.8
Básica incompleta,1.0
Media CH completa,1.0
Media CH incompleta,0.9
Media TP completa,1.1
Media TP incompleta,1.1
Sin educación formal,0.5
Superior técnica completa,1.4
Superior técnica incompleta,1.8


,n_smartphones_hogar
educ_jh,
Básica completa,3.1
Básica incompleta,3.2
Media CH completa,2.8
Media CH incompleta,3.1
Media TP completa,2.9
Media TP incompleta,3.5
Sin educación formal,2.8
Superior técnica completa,2.7
Superior técnica incompleta,2.9


...

,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
Permite comunicarse con otras personas,57.5,51.7,65.6,60.9,68.1,51.2,39.6,67.4,72.6,69.8,72.6
Permite tener más acceso a información,51.6,49.1,64.9,56.1,65.5,44.5,47.1,68.6,68.2,75.4,68.2
"Permite acceder a entretenimiento a través de streaming de películas, series y música",48.0,48.7,49.5,49.2,55.7,52.6,24.8,57.9,55.1,69.0,65.2
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",41.4,33.7,49.0,47.5,59.7,37.9,25.8,64.6,53.6,66.1,62.6
Apoyo a la educación propia o de hijos / nietos / parientes,40.5,45.4,54.4,55.3,54.6,57.4,27.8,53.6,71.5,48.3,59.6
Permite realizar trabajo o estudios desde casa,33.6,26.3,36.5,41.4,48.3,45.9,48.4,59.3,53.5,66.5,64.0
Permite participar en redes sociales y comunidades en línea,32.7,32.4,45.6,41.6,49.2,37.0,11.8,55.9,51.0,57.3,63.5
Proporciona acceso a noticias y actualidad,32.3,29.1,39.4,42.6,51.2,29.7,12.9,55.9,48.9,57.5,52.0
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,27.3,20.8,33.6,27.5,48.8,32.4,4.4,51.2,51.7,57.3,57.2
Permite acceder a juegos y otros medios de entretención,25.1,35.5,32.4,32.5,36.6,25.7,21.3,43.1,47.7,47.5,37.9


,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
Permite comunicarse con otras personas,83.2,82.9,83.7,83.4,76.4,60.8,84.8,82.9,82.1,85.4,79.6
Permite tener más acceso a información,44.4,42.5,55.8,50.0,58.5,43.0,23.0,58.1,45.5,58.5,59.0
Permite participar en redes sociales y comunidades en línea,36.5,30.8,47.3,42.2,52.8,36.4,17.4,51.9,41.9,55.2,46.2
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",31.4,25.9,42.1,34.7,54.3,29.1,23.1,50.2,40.2,51.7,51.1
"Permite acceder a entretenimiento a través de streaming de películas, series y música",26.4,27.8,32.9,32.7,46.1,33.7,40.2,43.9,42.9,46.3,43.9
Permite realizar trabajo o estudios desde casa,23.0,19.4,24.6,19.6,36.1,34.4,3.7,39.4,29.8,45.6,46.0
Apoyo a la educación propia o de hijos / nietos / parientes,22.9,20.0,30.4,28.3,33.8,28.8,12.0,37.5,23.1,31.5,34.6
Proporciona acceso a noticias y actualidad,21.7,19.9,29.6,25.7,44.4,25.4,9.3,38.6,38.1,50.0,36.5
Permite acceder a juegos y otros medios de entretención,18.2,19.3,28.2,22.1,35.9,19.4,8.6,33.1,25.9,33.2,40.9
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,17.0,16.2,27.6,20.4,40.7,19.9,5.1,40.7,36.5,39.9,35.2


,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
"Supervisión y/o monitoreo de uso de Internet (en presencia de adulto, verificar historial de navegación, etc.)",44.1,31.9,36.4,19.2,45.1,55.5,22.2,34.3,38.3,55.4,62.3
"Se acuerdan reglas sobre el uso de Internet (horarios, tiempo y condiciones)",36.6,47.6,46.8,58.5,53.4,49.3,50.8,47.5,49.9,52.2,75.9
"Educar a los niños sobre el uso seguro y responsable de Internet (ej: no publicar sus nombres completos en redes sociales, qué se puede hacer, cómo detectar relaciones / interacciones inseguras, etc.)",36.3,34.8,30.0,28.3,44.0,37.6,26.1,29.9,28.3,44.4,58.8
Restricción o bloqueo de ciertas aplicaciones o plataformas en línea.,31.7,29.1,20.0,16.6,27.7,40.9,12.5,23.0,27.6,28.8,32.5
Filtro de contenido para limitar el acceso a sitios web inapropiados.,24.4,21.9,16.8,15.2,22.3,27.5,12.5,25.3,20.5,21.4,36.1
Restricción de acceso a redes sociales y mensajería instantánea.,24.4,28.3,13.5,9.5,13.3,30.1,12.5,16.0,19.5,25.4,40.4
"Permitir acceso a Internet sólo desde lugares comunes / públicas de la casa (living, comedor, etc.)",24.2,15.1,15.1,19.7,20.1,27.5,34.6,19.0,19.2,14.6,15.0
Uso de controles parentales en los dispositivos conectados a internet.,22.1,18.4,18.2,12.7,17.8,27.8,12.5,25.7,47.8,24.3,54.8
No es necesario tener medidas de protección,20.1,21.0,24.7,25.8,16.8,9.6,13.6,28.8,11.1,23.6,4.7
Restricción de descargas o instalación de aplicaciones sin permiso.,19.6,18.2,16.6,15.9,24.8,27.5,12.5,23.0,18.9,28.6,40.6


,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
Smartphone,66.4,71.6,64.4,66.5,71.5,75.7,45.3,54.2,79.1,54.6,78.6
Computador / Notebook,36.9,26.0,37.1,36.9,48.8,57.5,44.2,34.1,22.6,40.4,50.6
"Tablet (Ipad, Galaxy TAB, Xperia Tablet, HP palm touchpad, etc.)",22.5,17.3,16.5,19.1,21.8,27.8,23.2,31.8,12.3,29.5,22.5
"Consola de juegos (WII, Nintendo, PS-3, PS-4, PS-5, XBOX,etc.)",18.7,29.5,20.0,30.4,32.4,47.9,9.7,25.1,20.7,33.8,33.0


,Básica completa,Básica incompleta,Media CH completa,Media CH incompleta,Media TP completa,Media TP incompleta,Sin educación formal,Superior técnica completa,Superior técnica incompleta,Superior universitaria completa,Superior universitaria incompleta
No he podido detectar ninguna situación,74.9,86.9,74.9,81.1,71.5,75.3,100.0,71.1,70.2,75.1,56.8
"Ser molestado, recibir hostilidad u hostigamiento por Internet (cyber bullying)",14.7,1.3,10.4,3.8,11.9,3.1,0.0,6.6,4.9,6.5,2.2
Intentos de engaños estafas o robo de información personal,11.3,7.5,15.6,12.2,18.9,19.0,0.0,21.0,5.7,17.0,29.1
NS/NR,3.9,4.8,5.9,2.3,5.6,4.1,0.0,6.1,15.7,4.2,12.6
"Intentos de captación o seducción, relaciones inapropiadas con menores de edad",3.2,0.0,3.7,2.5,9.4,3.8,0.0,3.4,3.5,2.1,1.4


### POR OCUPACIÓN JEFE DE HOGAR

In [26]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='ocupacion_jh', factor='fe_hogar', estilo=True))

ocupacion_jh,Trabajos ocasionales e informales,Oficio menor - obrero no calificado,Obrero calificado - microempresario,Empleado medio - técnico - prof. independiente,Ejecutivo medio - prof. universitario,Alto ejecutivo - empresario - directivo
acceso_internet_hogar,,,,,,
Sí,94.9,91.1,95.7,98.9,99.7,100.0
No,5.1,8.9,4.3,1.1,0.3,0.0


ocupacion_jh,Trabajos ocasionales e informales,Oficio menor - obrero no calificado,Obrero calificado - microempresario,Empleado medio - técnico - prof. independiente,Ejecutivo medio - prof. universitario,Alto ejecutivo - empresario - directivo
n_computadores_hogar,,,,,,
0.000000,44.0,41.9,41.8,21.1,6.7,0.0
1.000000,33.0,39.9,38.4,40.4,27.5,32.0
2.000000,14.7,13.8,13.8,24.6,38.2,27.9
3.000000,6.8,3.7,4.2,9.0,18.8,4.5
4.000000,0.3,0.4,1.2,2.4,6.3,31.3
5.000000,0.1,0.1,0.2,1.5,2.4,1.5
6.000000,0.9,0.0,0.1,0.4,0.0,0.0
7.000000,0.2,0.0,0.0,0.4,0.0,0.0
8.000000,0.0,0.0,0.4,0.3,0.0,1.5


ocupacion_jh,Trabajos ocasionales e informales,Oficio menor - obrero no calificado,Obrero calificado - microempresario,Empleado medio - técnico - prof. independiente,Ejecutivo medio - prof. universitario,Alto ejecutivo - empresario - directivo
n_smartphones_hogar,,,,,,
0.000000,0.4,0.4,0.2,0.4,0.0,0.0
1.000000,22.1,20.0,17.6,18.2,18.0,16.4
2.000000,28.1,30.3,28.1,29.1,31.7,30.3
3.000000,18.9,24.5,24.9,24.9,22.1,10.6
4.000000,17.7,14.3,15.8,16.4,21.4,36.9
5.000000,4.8,4.9,7.7,6.3,3.3,0.0
6.000000,3.5,3.4,2.5,2.3,2.5,4.6
7.000000,2.5,0.6,1.8,1.2,0.9,0.0
8.000000,0.5,1.3,0.1,0.4,0.0,1.2


,n_computadores_hogar
ocupacion_jh,
Trabajos ocasionales e informales,1.0
Oficio menor - obrero no calificado,1.0
Obrero calificado - microempresario,0.9
Empleado medio - técnico - prof. independiente,1.4
Ejecutivo medio - prof. universitario,2.0
Alto ejecutivo - empresario - directivo,2.6


,n_smartphones_hogar
ocupacion_jh,
Trabajos ocasionales e informales,2.9
Oficio menor - obrero no calificado,2.8
Obrero calificado - microempresario,3.0
Empleado medio - técnico - prof. independiente,2.9
Ejecutivo medio - prof. universitario,2.8
Alto ejecutivo - empresario - directivo,2.9


...

,Alto ejecutivo - empresario - directivo,Ejecutivo medio - prof. universitario,Empleado medio - técnico - prof. independiente,Obrero calificado - microempresario,Oficio menor - obrero no calificado,Trabajos ocasionales e informales
Permite tener más acceso a información,84.0,73.6,70.4,58.8,56.0,55.4
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",80.2,67.2,61.4,46.2,49.0,39.6
Permite comunicarse con otras personas,75.9,71.7,66.9,63.5,62.1,56.3
Permite realizar trabajo o estudios desde casa,71.7,71.6,57.4,35.6,37.7,36.0
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,71.4,60.1,50.9,32.1,33.1,25.1
Proporciona acceso a noticias y actualidad,69.9,59.6,52.3,39.5,36.8,30.8
"Permite acceder a entretenimiento a través de streaming de películas, series y música",67.1,70.1,61.1,50.4,52.8,40.9
Apoyo a la educación propia o de hijos / nietos / parientes,58.2,49.4,52.2,49.6,60.2,52.2
Permite participar en redes sociales y comunidades en línea,58.2,62.7,54.6,41.5,42.2,32.1
Permite acceder a servicios de telemedicina o consultas médicas en línea,51.8,55.0,40.0,27.9,28.5,29.9


,Alto ejecutivo - empresario - directivo,Ejecutivo medio - prof. universitario,Empleado medio - técnico - prof. independiente,Obrero calificado - microempresario,Oficio menor - obrero no calificado,Trabajos ocasionales e informales
Permite comunicarse con otras personas,84.3,85.4,81.0,83.0,84.2,77.4
Permite tener más acceso a información,72.2,61.2,56.6,49.5,50.4,52.3
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",56.3,52.2,49.6,38.3,38.2,33.9
Proporciona acceso a noticias y actualidad,56.3,52.2,41.2,26.2,27.9,26.6
Permite realizar trabajo o estudios desde casa,53.4,53.8,36.8,23.1,29.0,25.3
Permite participar en redes sociales y comunidades en línea,44.7,59.2,50.4,42.5,44.2,37.4
"Permite acceder a entretenimiento a través de streaming de películas, series y música",43.8,51.3,44.2,30.4,35.5,28.9
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,40.6,45.4,38.6,21.4,26.6,22.3
Permite realizar negocios o manejar una empresa familiar.,36.1,32.4,21.5,16.7,14.5,16.7
Permite acceder a servicios de telemedicina o consultas médicas en línea,33.9,38.2,27.6,17.3,17.3,21.7


,Alto ejecutivo - empresario - directivo,Ejecutivo medio - prof. universitario,Empleado medio - técnico - prof. independiente,Obrero calificado - microempresario,Oficio menor - obrero no calificado,Trabajos ocasionales e informales
"Se acuerdan reglas sobre el uso de Internet (horarios, tiempo y condiciones)",42.3,59.3,50.9,49.3,48.3,40.5
No es necesario tener medidas de protección,35.7,19.8,24.6,24.1,18.3,14.2
"Educar a los niños sobre el uso seguro y responsable de Internet (ej: no publicar sus nombres completos en redes sociales, qué se puede hacer, cómo detectar relaciones / interacciones inseguras, etc.)",30.8,47.0,39.1,32.2,32.3,32.0
"Supervisión y/o monitoreo de uso de Internet (en presencia de adulto, verificar historial de navegación, etc.)",30.1,63.7,44.0,36.7,34.2,32.2
Instalación de filtros de Internet (software de control parental),23.6,34.0,26.1,20.5,18.9,13.8
Uso de controles parentales en los dispositivos conectados a internet.,19.3,29.9,25.8,20.2,12.0,21.7
Restricción de descargas o instalación de aplicaciones sin permiso.,18.6,33.3,24.5,19.0,15.7,19.2
Restricción o bloqueo de ciertas aplicaciones o plataformas en línea.,13.9,28.4,26.2,25.7,24.6,21.6
Restricción de acceso a redes sociales y mensajería instantánea.,12.8,27.4,23.2,15.4,18.3,14.5
NS/NR,4.8,0.0,4.0,2.0,2.3,17.1


,Alto ejecutivo - empresario - directivo,Ejecutivo medio - prof. universitario,Empleado medio - técnico - prof. independiente,Obrero calificado - microempresario,Oficio menor - obrero no calificado,Trabajos ocasionales e informales
Smartphone,86.0,59.8,58.0,66.1,69.7,70.4
Computador / Notebook,57.6,48.9,35.3,36.5,35.7,45.3
"Consola de juegos (WII, Nintendo, PS-3, PS-4, PS-5, XBOX,etc.)",32.5,45.1,27.2,21.8,25.3,27.0
"Tablet (Ipad, Galaxy TAB, Xperia Tablet, HP palm touchpad, etc.)",21.9,33.1,26.8,17.3,19.2,20.9


,Alto ejecutivo - empresario - directivo,Ejecutivo medio - prof. universitario,Empleado medio - técnico - prof. independiente,Obrero calificado - microempresario,Oficio menor - obrero no calificado,Trabajos ocasionales e informales
No he podido detectar ninguna situación,76.6,69.0,71.2,79.5,80.5,71.8
"Ser molestado, recibir hostilidad u hostigamiento por Internet (cyber bullying)",18.6,2.5,10.2,6.9,6.0,9.6
NS/NR,4.8,5.7,6.5,3.0,4.2,9.8
Intentos de engaños estafas o robo de información personal,3.2,22.3,20.0,13.3,9.5,13.3
"Intentos de captación o seducción, relaciones inapropiadas con menores de edad",0.0,3.3,3.8,3.7,2.2,2.4


### POR SEXO

In [27]:
display(dstats(df, 'acceso_internet_hogar', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_computadores_hogar', tipo='promedio', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'n_smartphones_hogar', tipo='promedio', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_fijo', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(dstats(df, 'tipo_acceso_mas_usado', tipo='cruzada', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P3', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P4', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P6_1', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P7', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P9', cruce='sexo', factor='fe_hogar', estilo=True))
display(analizar_rm_cruce('P8_3', cruce='sexo', factor='fe_hogar', estilo=True))

sexo,Hombre,Mujer
acceso_internet_hogar,,
Sí,96.9,96.3
No,3.1,3.7


sexo,Hombre,Mujer
n_computadores_hogar,,
0.000000,29.0,32.6
1.000000,36.2,38.3
2.000000,22.5,18.5
3.000000,7.7,7.6
4.000000,2.6,1.9
5.000000,0.8,0.9
6.000000,0.6,0.0
7.000000,0.3,0.0
8.000000,0.3,0.2


sexo,Hombre,Mujer
n_smartphones_hogar,,
0.000000,0.2,0.3
1.000000,21.1,16.6
2.000000,28.5,29.6
3.000000,22.9,24.5
4.000000,15.6,17.9
5.000000,5.8,6.2
6.000000,3.4,2.1
7.000000,1.5,1.4
8.000000,0.2,0.6


,n_computadores_hogar
sexo,
Hombre,1.3
Mujer,1.2


,n_smartphones_hogar
sexo,
Hombre,2.9
Mujer,2.9


...

,Hombre,Mujer
Permite tener más acceso a información,66.4,63.1
Permite comunicarse con otras personas,64.6,65.6
"Permite acceder a entretenimiento a través de streaming de películas, series y música",60.4,52.8
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",55.2,53.8
Permite participar en redes sociales y comunidades en línea,49.4,47.5
Permite realizar trabajo o estudios desde casa,48.8,49.0
Proporciona acceso a noticias y actualidad,47.6,44.6
Apoyo a la educación propia o de hijos / nietos / parientes,44.3,58.4
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,44.0,41.0
Permite acceder a juegos y otros medios de entretención,38.9,36.1


,Hombre,Mujer
Permite comunicarse con otras personas,81.3,82.8
Permite tener más acceso a información,55.7,52.2
Permite participar en redes sociales y comunidades en línea,47.7,45.8
"Permite realizar trámites personales como revisar cuentas bancarias, realizar transferencias, solicitar certificados y pagar cuentas",45.2,41.8
"Permite acceder a entretenimiento a través de streaming de películas, series y música",39.6,36.5
Proporciona acceso a noticias y actualidad,37.7,32.1
Permite realizar trabajo o estudios desde casa,34.1,30.6
Facilita la realización de compras en línea y acceso a plataformas de comercio electrónico.,33.2,28.6
Permite acceder a juegos y otros medios de entretención,31.4,26.5
Apoyo a la educación propia o de hijos / nietos / parientes,27.7,32.0


,Hombre,Mujer
"Se acuerdan reglas sobre el uso de Internet (horarios, tiempo y condiciones)",51.2,49.0
"Supervisión y/o monitoreo de uso de Internet (en presencia de adulto, verificar historial de navegación, etc.)",43.9,39.9
"Educar a los niños sobre el uso seguro y responsable de Internet (ej: no publicar sus nombres completos en redes sociales, qué se puede hacer, cómo detectar relaciones / interacciones inseguras, etc.)",36.8,35.7
No es necesario tener medidas de protección,23.8,21.3
Restricción o bloqueo de ciertas aplicaciones o plataformas en línea.,23.1,26.8
Restricción de descargas o instalación de aplicaciones sin permiso.,22.9,21.7
Instalación de filtros de Internet (software de control parental),22.6,23.1
Uso de controles parentales en los dispositivos conectados a internet.,22.2,22.6
Filtro de contenido para limitar el acceso a sitios web inapropiados.,21.4,21.3
Restricción de acceso a redes sociales y mensajería instantánea.,17.7,20.5


,Hombre,Mujer
Smartphone,64.5,63.3
Computador / Notebook,41.2,37.1
"Consola de juegos (WII, Nintendo, PS-3, PS-4, PS-5, XBOX,etc.)",30.5,25.4
"Tablet (Ipad, Galaxy TAB, Xperia Tablet, HP palm touchpad, etc.)",19.7,24.6


,Hombre,Mujer
No he podido detectar ninguna situación,69.8,77.5
Intentos de engaños estafas o robo de información personal,21.1,13.3
"Ser molestado, recibir hostilidad u hostigamiento por Internet (cyber bullying)",11.9,5.7
NS/NR,4.8,5.7
"Intentos de captación o seducción, relaciones inapropiadas con menores de edad",4.0,3.0


# Habilidades (Personas)

## Habilidades por nivel

In [28]:
display(dstats(df,'hab_avanzadas', tipo='frecuencia', factor='fe_personas'))
display(dstats(df,'hab_intermedias', tipo='frecuencia', factor='fe_personas'))
display(dstats(df,'hab_basicas', tipo='frecuencia', factor='fe_personas'))

,porcentaje
hab_avanzadas,
No,57.2
Sí,42.8


,porcentaje
hab_intermedias,
No,14.4
Sí,85.6


,porcentaje
hab_basicas,
No,4.5
Sí,95.5


## Habilidades por tipo

In [29]:
display(dstats(df,'usa_productivas', tipo='frecuencia', factor='fe_personas'))
display(dstats(df,'usa_recreativas', tipo='frecuencia', factor='fe_personas'))
display(dstats(df,'usa_creativas', tipo='frecuencia', factor='fe_personas'))
display(dstats(df,'usa_comunicaciones', tipo='frecuencia', factor='fe_personas'))


,porcentaje
usa_productivas,
No,16.5
Sí,83.5


,porcentaje
usa_recreativas,
No,33.8
Sí,66.2


,porcentaje
usa_creativas,
No,36.9
Sí,63.1


,porcentaje
usa_comunicaciones,
No,5.4
Sí,94.6


## Personas sin habilidades

In [44]:
dstats(df, 'no_hab', tipo='frecuencia', factor='fe_personas', estilo=True)

,porcentaje
no_hab,
Al menos una habilidad,97.7
Sin habilidades,2.3


In [42]:
df.to_csv('datos_limpios.csv')